# Macro Regime Filter: Unemployment × Policy-Rate Cycle Backtest

**Thesis.** Through an expansion the policy rate rises while unemployment keeps grinding lower. Near the **peak** the labor market stops improving; a **recession** follows in which the Fed cuts hard (*rates collapse*) and unemployment spikes to its cycle high. When unemployment finally rolls over from that high, with rates at the floor, the **recovery** starts and the cycle repeats.

**Strategy.** Turn the cycle into a **confidence score** (0-100 %, reported in 10 % steps) that the expansion is intact, and size equity exposure from it, keeping the rest in cash (3-month T-bills).

| § | Section | Content |
|---|---|---|
| 1 | Setup | Parameters and plotting style |
| 2 | Data | FRED (public CSV, no key) and Yahoo Finance, cached in `data/` |
| 3 | Cycle features | Unemployment and fed funds measured inside a **10-quarter** window |
| 4 | Phase map | Expansion → Peak → Recession → Recovery, checked against NBER dates |
| 5 | In-sample market study | How the S&P 500 and Nasdaq-100 behaved in each phase |
| 6 | Confidence model | Walk-forward logistic model: *P(no contraction in the next 6 months)* |
| 7 | Allocation rules and engine | Proportional 50/50, tiered NDX/SPX/cash, drift-aware backtester |
| 8 | Strategies and optimization | User rules, grid-searched tiers (1986-2004 only), trend-filter extension, benchmarks |
| 9 | Results | CAGR, vol, Sharpe, Sortino, max DD, Calmar, Ulcer, VaR, beta, capture, exposure, PSR/DSR |
| 10 | Robustness | Lookback, horizon, costs, crisis windows, bootstrap significance, **look-ahead audit, placebo timing test, probability of backtest overfitting** |
| 11 | Research views | Does old data hurt? (expanding vs rolling vs time-decay training); tails, kurtosis and drawdowns |
| 12 | **Live** | Today's confidence with a 90 % bootstrap interval and the allocation for each rule |

**Bias controls**
* **Publication lag.** A month's unemployment and fed-funds print is only used from the end of the *following* month.
* **As-of labels.** At every refit the training labels are rebuilt from only what was published by then: unemployment up to last month, and NBER dating up to 12 months ago, because NBER announces turning points 4-21 months late.
* **Look-ahead audit.** §10e rebuilds the entire signal from data truncated at sample dates and requires an exact match.
* **Walk-forward.** The model is re-fit every 12 months on what was knowable at that time, so every backtest signal is out-of-sample.
* **Optimization hold-out.** Tier thresholds and band allocations are chosen on 1986-2004 and evaluated on 2005-today. The Deflated Sharpe Ratio penalizes the number of configurations tried.
* **Execution.** Fills at the close one trading day after the month-end signal, holdings drift between rebalances, 5 bps per unit of turnover, and idle cash earns the T-bill rate.

> **"10-quarter step confidence"** is implemented as: unemployment and rates are judged relative to their own **10-quarter (30-month) range**, and confidence is reported in **10 % steps**. Both are parameters (`LOOKBACK_Q`, `CONF_STEP`); §10 stress-tests the lookback.

*Research code, not investment advice.*

## 1 · Setup

In [ ]:
import io
import itertools
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from scipy import stats
from scipy.optimize import minimize
from scipy.special import expit
from sklearn.metrics import brier_score_loss, roc_auc_score
from IPython.display import display
import yfinance as yf

warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 30)
pd.set_option('display.max_rows', 120)

# ---- research parameters ---------------------------------------------------------
LOOKBACK_Q     = 10            # cycle window in quarters
LOOKBACK_M     = LOOKBACK_Q * 3
CONF_STEP      = 0.10          # confidence reported in 10 % steps
HORIZON_M      = 6             # model target: contraction (NBER peak -> unemployment high) within the next N months
NBER_LAG_M     = 12            # extra embargo before a contraction label counts as "known"
PUB_LAG_M      = 1             # macro print for month M is used from the end of month M+1
REFIT_EVERY_M  = 12            # walk-forward re-fit frequency (months)
TRAIN_WINDOW_M = None          # None = expanding window (all history); e.g. 240 = rolling 20-year window
TRAIN_HALF_LIFE_M = None       # None = equal weights; e.g. 180 = an observation's weight halves every 15 years
L2_C           = 0.5           # inverse L2 strength of the logistic model
Z_CLIP         = 3.0           # standardised inputs clipped to +/- this many sigmas (limits extrapolation)
MIN_RANGE      = 1.0           # pp floor on the 10-quarter range (stops /0 at the zero bound)

# Phase rules (pp unless stated); set a priori from economic reasoning, not fitted
PHASE_RULES = dict(
    ur_rising=0.20,    # 6m change in 3m-avg unemployment that counts as "labor market weakening"
    sahm=0.50,         # unemployment 3m-avg above its 12m low by this much (Sahm-style trigger)
    ff_falling=0.25,   # 6m fed-funds cut that counts as "easing" (one standard Fed move)
    ff_collapse=1.00,  # fed funds this far below its 10-quarter high counts as a rate collapse
    ur_rollover=0.30,  # unemployment 3m-avg this far below its 6m high = rolling over from a peak
    ur_high=0.50,      # unemployment in the upper half of its 10-quarter range = "elevated"
    peak_ur=0.20,      # Peak zone: unemployment in the bottom 20 % of its range ...
    peak_ff=0.80,      # ... while fed funds is in the top 20 % of its range
)

BACKTEST_START = '1986-01'     # first decision month (Nasdaq-100 history starts Oct-1985)
DESIGN_END     = '2004-12'     # optimization window ends here; 2005+ is the untouched test set
COST_BPS       = 5.0           # cost per unit of one-way turnover (ETF / futures-like)
EXEC_LAG_DAYS  = 1             # fill at the close N trading days after the month-end signal
TREND_SMA_M    = 10            # trend-filter extension: S&P 500 vs its 10-month SMA
OPT_OBJECTIVE  = 'Sharpe'      # objective for the tier optimizer ('Sharpe', 'Calmar', 'Sortino', 'CAGR')

# The user's tier rule: >= 70 % -> 100 % Nasdaq-100, 40-69 % -> 100 % S&P 500, < 40 % -> cash
USER_TIERS = dict(hi=0.70, lo=0.40, top=(0.0, 1.0), mid=(1.0, 0.0), low=(0.0, 0.0))

DATA_DIR = Path('data')
REFRESH_DATA = True            # False = only use the cached CSVs in data/

# ---- plotting style ----------------------------------------------------------------
C = dict(blue='#2a78d6', orange='#eb6834', aqua='#1baf7a', yellow='#eda100', red='#e34948',
         violet='#4a3aa7', ink='#0b0b0b', ink2='#52514e', muted='#8f8e88', light='#bdbcb6',
         grid='#e8e7e3', nber='#dcdbd5')
PHASES = ['Expansion', 'Peak', 'Recession', 'Recovery']
PHASE_COLORS = {'Expansion': C['aqua'], 'Peak': C['yellow'], 'Recession': C['red'], 'Recovery': C['blue']}
plt.rcParams.update({
    'figure.figsize': (12, 4.2), 'figure.dpi': 110, 'figure.facecolor': 'white',
    'axes.facecolor': 'white', 'axes.edgecolor': C['light'], 'axes.labelcolor': C['ink2'],
    'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True, 'axes.axisbelow': True,
    'axes.titleweight': 'bold', 'axes.titlesize': 11, 'axes.titlelocation': 'left',
    'grid.color': C['grid'], 'grid.linewidth': 0.6, 'grid.linestyle': '-',
    'xtick.color': C['ink2'], 'ytick.color': C['ink2'], 'legend.frameon': False,
    'lines.linewidth': 1.6, 'font.size': 9.5,
})
print('pandas', pd.__version__, '| numpy', np.__version__, '| yfinance', yf.__version__)

## 2 · Data
* **FRED** via the public `fredgraph.csv` endpoint (no API key): `UNRATE`, `FEDFUNDS`, `TB3MS` (cash yield), and `USREC`, the NBER recession flag. `USREC` is used **only** for training labels and validation and is never a live input.
* **Yahoo Finance.**
  * S&P 500 sleeve: `SPY` (1993+), then `^SP500TR` (1988+), then `^GSPC` with a 3.5 % p.a. dividend approximation before 1988.
  * Nasdaq-100 sleeve: `QQQ` (1999+), then `^NDX` with a 0.5 % p.a. dividend approximation.
  * A fully collateralized ES/NQ futures position earns roughly the index total return, because the futures' financing cost is offset by the T-bill yield on the collateral. These sleeves therefore stand in for ES and NQ.
* Every download is cached in `data/`. If a download fails, the cached copy is used and a warning is printed.

In [ ]:
import os
import time

DATA_DIR.mkdir(exist_ok=True)

# Optional but most reliable: a free FRED API key from https://fredaccount.stlouisfed.org/apikeys
# Paste it here or set the FRED_API_KEY environment variable. Without a key the public CSV endpoint is used.
FRED_API_KEY = os.environ.get('FRED_API_KEY', '')

BROWSER_HEADERS = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 '
                                 '(KHTML, like Gecko) Chrome/124.0 Safari/537.36',
                   'Accept': 'text/csv,text/plain,*/*'}
_fred_unreachable = False      # set after every source fails once, so later series skip straight to the cache


def _cache(kind, name):
    return DATA_DIR / f"{kind}_{name.replace('^', '_')}.csv"


def _read_series_csv(src):
    """Two-column CSV (date, value). Also reads files saved straight from the FRED website ('.' = missing)."""
    df = pd.read_csv(src)
    s = pd.Series(pd.to_numeric(df.iloc[:, 1], errors='coerce').values, index=pd.to_datetime(df.iloc[:, 0]))
    return s.dropna().sort_index()


def _parse_fred_csv(text, series_id):
    first = text.lstrip().split('\n', 1)[0]
    if series_id not in first:                       # e.g. an HTML error / bot-check page instead of the CSV
        raise ValueError(f'unexpected response: {first[:80]!r}')
    return _read_series_csv(io.StringIO(text))


def _fred_api(series_id):
    import requests
    r = requests.get('https://api.stlouisfed.org/fred/series/observations', timeout=30,
                     params={'series_id': series_id, 'api_key': FRED_API_KEY, 'file_type': 'json'})
    r.raise_for_status()
    obs = pd.DataFrame(r.json()['observations'])
    return pd.Series(pd.to_numeric(obs['value'], errors='coerce').values, index=pd.to_datetime(obs['date'])).dropna()


def _fred_csv(series_id):
    import requests                                  # ships its own CA bundle (fixes most SSL errors)
    r = requests.get('https://fred.stlouisfed.org/graph/fredgraph.csv', params={'id': series_id},
                     headers=BROWSER_HEADERS, timeout=30)
    r.raise_for_status()
    return _parse_fred_csv(r.text, series_id)


def _fred_csv_browser(series_id):
    from curl_cffi import requests as creq           # installed with yfinance; looks like a real Chrome browser
    r = creq.get(f'https://fred.stlouisfed.org/graph/fredgraph.csv?id={series_id}', impersonate='chrome', timeout=30)
    r.raise_for_status()
    return _parse_fred_csv(r.text, series_id)


def fetch_fred(series_id, refresh=REFRESH_DATA):
    """FRED series: official API (if FRED_API_KEY) -> public CSV -> public CSV via browser TLS -> data/ cache."""
    global _fred_unreachable
    path = _cache('fred', series_id)
    if refresh and not _fred_unreachable:
        sources = ([('FRED API', _fred_api)] if FRED_API_KEY else []) + \
                  [('fredgraph.csv', _fred_csv), ('fredgraph.csv/browser', _fred_csv_browser)]
        errors = []
        for name, fn in sources:
            try:
                s = fn(series_id)
                if s.empty:
                    raise ValueError('empty series')
                s.rename_axis('date').rename('value').to_csv(path)
                return s.rename(series_id)
            except Exception as e:
                errors.append(f'{name}: {type(e).__name__}: {str(e)[:150]}')
        _fred_unreachable = True
        print(f'  ! FRED {series_id}: every source failed\n      ' + '\n      '.join(errors))
    if path.exists():
        print(f'    FRED {series_id}: using cached {path}')
        return _read_series_csv(path).rename(series_id)
    raise FileNotFoundError(
        f'FRED:{series_id} could not be downloaded (see the errors printed above) and there is no cached copy.\n'
        f'Fix one of these, then re-run this cell:\n'
        f'  1) get a free key at https://fredaccount.stlouisfed.org/apikeys and set FRED_API_KEY in this cell, or\n'
        f'  2) open https://fred.stlouisfed.org/graph/fredgraph.csv?id={series_id} in a browser and save it as\n'
        f'     {path.resolve()}  (repeat for {", ".join(FRED_IDS)})')


def fetch_yf(ticker, refresh=REFRESH_DATA, attempts=3):
    """Daily split- and dividend-adjusted close from Yahoo Finance, cached to data/."""
    path = _cache('yf', ticker)
    if refresh:
        err = None
        for i in range(attempts):
            try:
                df = yf.download(ticker, period='max', auto_adjust=True, progress=False, threads=False)
                if isinstance(df.columns, pd.MultiIndex):
                    df.columns = df.columns.get_level_values(0)
                s = df['Close'].dropna() if 'Close' in df else pd.Series(dtype=float)
                if s.empty:
                    raise ValueError('no rows returned (rate limit or outdated yfinance: pip install -U yfinance)')
                idx = pd.to_datetime(s.index)
                s.index = (idx.tz_localize(None) if idx.tz is not None else idx).normalize()
                s.rename_axis('Date').rename('Close').to_csv(path)
                return s.rename(ticker)
            except Exception as e:
                err = f'{type(e).__name__}: {str(e)[:150]}'
                time.sleep(2 * (i + 1))
        print(f'  ! Yahoo {ticker}: download failed ({err})')
    if not path.exists():
        return None
    print(f'    Yahoo {ticker}: using cached {path}')
    return _read_series_csv(path).rename(ticker)


FRED_IDS = ['UNRATE', 'FEDFUNDS', 'TB3MS', 'USREC']
SLEEVES = {'SPX': ['SPY', '^SP500TR', '^GSPC'],      # priority order: ETF > TR index > price index
           'NDX': ['QQQ', '^NDX']}
APPROX_DIV_YIELD = {'^GSPC': 0.035, '^NDX': 0.005}   # only added to price-only history


def load_macro(refresh=REFRESH_DATA):
    m = pd.concat({k: fetch_fred(k, refresh) for k in FRED_IDS}, axis=1)
    m.index = pd.to_datetime(m.index).to_period('M')
    return m.sort_index()


def spliced_returns(tickers, prices):
    """Daily total returns: each higher-priority ticker takes over from its first date."""
    out = None
    for t in reversed(tickers):                       # lowest priority / longest history first
        s = prices.get(t)
        if s is None or len(s) < 2:
            continue
        r = s.sort_index().pct_change().dropna() + APPROX_DIV_YIELD.get(t, 0.0) / 252
        out = r if out is None else pd.concat([out[out.index < r.index[0]], r])
    if out is None:
        raise RuntimeError(f'no Yahoo data for any of {tickers}; see the download errors above (try: pip install -U yfinance)')
    return out


macro_raw = load_macro()
prices = {t: fetch_yf(t) for lst in SLEEVES.values() for t in lst}

spx_r = spliced_returns(SLEEVES['SPX'], prices)
cal = spx_r.index                                         # master trading calendar
ndx_lvl = (1 + spliced_returns(SLEEVES['NDX'], prices)).cumprod()
ndx_lvl = ndx_lvl.reindex(cal.union(ndx_lvl.index)).ffill().reindex(cal)
ndx_r = ndx_lvl / ndx_lvl.shift(1) - 1

tb = macro_raw['TB3MS'].dropna()
tb_prev = tb.shift(1).reindex(pd.period_range(tb.index[0], cal[-1].to_period('M'), freq='M')).ffill()
cash_r = pd.Series(((1 + tb_prev / 100) ** (1 / 252) - 1).reindex(cal.to_period('M')).values, index=cal)   # last month's bill yield

rets = pd.DataFrame({'SPX': spx_r, 'NDX': ndx_r, 'CASH': cash_r}).dropna(subset=['SPX', 'CASH'])
ASSETS = list(rets.columns)
mret = (1 + rets).groupby(rets.index.to_period('M')).prod(min_count=1) - 1      # monthly sleeve returns

spx_px = prices['^GSPC'] if prices.get('^GSPC') is not None else (1 + spx_r).cumprod()
spx_px_m = spx_px.groupby(spx_px.index.to_period('M')).last()                   # month-end price (trend filter)

coverage = pd.DataFrame(
    [(f'FRED {k}', macro_raw[k].first_valid_index(), macro_raw[k].last_valid_index()) for k in FRED_IDS]
    + [(f'Yahoo {t}', s.index[0].date() if s is not None else 'missing',
        s.index[-1].date() if s is not None else 'missing') for t, s in prices.items()],
    columns=['series', 'first', 'last']).set_index('series')
display(coverage)
print(f"Daily return panel: {rets.index[0].date()} -> {rets.index[-1].date()}  "
      f"(NDX from {rets['NDX'].first_valid_index().date()})")

## 3 · Cycle features (10-quarter window)
All features use only the unemployment rate and the fed-funds rate, the two variables in the thesis.

| Feature | Definition | Reads as |
|---|---|---|
| `ur_trend` | 6-month change of 3-month-avg unemployment | labor market improving (<0) or weakening (>0) |
| `ur_gap` | 3m-avg unemployment minus its 12-month low | Sahm-style recession trigger |
| `ur_off_high` | 3m-avg unemployment minus its 6-month high | rolling over from a cycle peak (<0) |
| `ur_pos` | position of unemployment inside its **10-quarter** min-max range | 0 = cycle low, 1 = cycle high |
| `ff_trend` | 6-month change in fed funds | tightening (>0) or easing (<0) |
| `ff_pos` | position of fed funds inside its **10-quarter** range | 0 = at the floor, 1 = at the top |
| `ff_off_peak` | fed funds minus its **10-quarter** high | size of the rate collapse |

Features are computed on the month a print *describes* and then moved forward by `PUB_LAG_M` to the month in which they can actually be traded.

In [ ]:
def build_features(m, lookback_m=LOOKBACK_M):
    ur, ff = m['UNRATE'], m['FEDFUNDS']
    ur3 = ur.rolling(3).mean()
    f = pd.DataFrame(index=m.index)
    f['ur'], f['ff'] = ur, ff
    f['ur_trend'] = ur3 - ur3.shift(6)
    f['ur_gap'] = ur3 - ur3.rolling(12).min()
    f['ur_off_high'] = ur3 - ur3.rolling(6).max()
    lo, hi = ur.rolling(lookback_m).min(), ur.rolling(lookback_m).max()
    f['ur_pos'] = (ur - lo) / np.maximum(hi - lo, MIN_RANGE)
    f['ff_trend'] = ff - ff.shift(6)
    flo, fhi = ff.rolling(lookback_m).min(), ff.rolling(lookback_m).max()
    f['ff_pos'] = (ff - flo) / np.maximum(fhi - flo, MIN_RANGE)
    f['ff_off_peak'] = ff - fhi
    return f.dropna()


def to_decision_time(feats, lag=PUB_LAG_M):
    """Re-index from 'month described' to 'month in which the print is usable'."""
    X = feats.copy()
    X.index = X.index + lag
    return X


feats = build_features(macro_raw)
X = to_decision_time(feats)
print(f'Features available for decisions {X.index[0]} -> {X.index[-1]}  ({len(X)} months)')
display(X.tail(6).round(2))

## 4 · Phase map: Expansion → Peak → Recession → Recovery
Rules are evaluated in priority order and mirror the thesis directly:

1. **Recovery (trough).** Unemployment is elevated in its 10-quarter range *and* rolling over from its high.
2. **Recession.** The labor market is weakening *and* rates are being cut or have collapsed.
3. **Peak (late cycle).** The labor market is weakening while policy is still tight, *or* unemployment is at its cycle low while fed funds is at its cycle high, *or* unemployment is drifting up into the top of its range.
4. **Recovery (early cycle).** Unemployment is still elevated but flat or falling.
5. **Expansion.** Everything else: unemployment falling or low, rates rising or normalizing.

The NBER flag is overlaid only to check the map. It is not an input.

In [ ]:
def classify_phase(X, r=PHASE_RULES):
    labor_weak = (X['ur_trend'] > r['ur_rising']) | (X['ur_gap'] >= r['sahm'])
    easing = (X['ff_trend'] < -r['ff_falling']) | (X['ff_off_peak'] <= -r['ff_collapse'])
    rolling_over = X['ur_off_high'] <= -r['ur_rollover']
    elevated = X['ur_pos'] >= r['ur_high']
    peak_zone = (X['ur_pos'] <= r['peak_ur']) & (X['ff_pos'] >= r['peak_ff'])
    ur_up = X['ur_trend'] > 0
    phase = np.select(
        [rolling_over & elevated, labor_weak & easing, (labor_weak & ~easing) | peak_zone | (elevated & ur_up),
         elevated & ~ur_up],
        ['Recovery', 'Recession', 'Peak', 'Recovery'], default='Expansion')
    return pd.Series(phase, index=X.index, name='phase')


def spells(s):
    """Contiguous runs of equal values -> list of (first period, last period, value)."""
    g = (s != s.shift()).cumsum()
    return [(grp.index[0], grp.index[-1], grp.iloc[0]) for _, grp in s.groupby(g)]


def shade(ax, s, colors, alpha=0.25):
    for a, b, v in spells(s.dropna()):
        if v in colors:
            ax.axvspan(a.start_time, (b + 1).start_time, color=colors[v], alpha=alpha, lw=0)


usrec = macro_raw['USREC']


def shade_nber(ax, start=None, alpha=0.6):
    shade(ax, usrec.loc[start:], {1: C['nber']}, alpha)


phase = classify_phase(X)

fig, axes = plt.subplots(3, 1, figsize=(12, 8.5), sharex=True, gridspec_kw=dict(height_ratios=[3, 0.5, 3]))
ax = axes[0]
shade_nber(ax, X.index[0])
m = macro_raw.loc[X.index[0] - PUB_LAG_M:]
ax.plot(m.index.to_timestamp(), m['UNRATE'], color=C['red'], label='Unemployment rate')
ax.plot(m.index.to_timestamp(), m['FEDFUNDS'], color=C['blue'], label='Fed funds rate')
ax.set_ylabel('%')
ax.set_title('Unemployment and the policy rate (grey = NBER recession)')
ax.legend(loc='upper left', ncol=2)
ax = axes[1]
shade(ax, phase, PHASE_COLORS, alpha=0.9)
ax.set_yticks([])
ax.grid(False)
ax.set_title('Phase (as known in real time)')
ax = axes[2]
shade(ax, phase, PHASE_COLORS, alpha=0.18)
lvl = (1 + mret['SPX'].loc[X.index[0]:]).cumprod()
ax.semilogy(lvl.index.to_timestamp(), lvl, color=C['ink'], lw=1.3)
ax.set_title('S&P 500 total return (log) shaded by phase')
fig.legend(handles=[Patch(color=PHASE_COLORS[p], label=p) for p in PHASES], loc='lower center', ncol=4,
           bbox_to_anchor=(0.5, -0.01))
fig.tight_layout(rect=(0, 0.03, 1, 1))
plt.show()

chk = pd.DataFrame({'phase': phase, 'nber': usrec.reindex(phase.index)}).dropna()
validation = pd.DataFrame({
    'months': chk.groupby('phase').size(),
    '% of time': chk.groupby('phase').size() / len(chk),
    '% of months in NBER recession': chk.groupby('phase')['nber'].mean(),
    '% of all NBER months in this phase': chk[chk.nber == 1].groupby('phase').size() / (chk.nber == 1).sum(),
}).reindex(PHASES).fillna(0)
display(validation.style.format({'months': '{:.0f}', '% of time': '{:.1%}',
                                 '% of months in NBER recession': '{:.1%}',
                                 '% of all NBER months in this phase': '{:.1%}'}))

### Event study: the thesis around NBER business-cycle peaks
Each thin line is one recession and the bold line is the median, with month 0 = NBER peak. This chart is *descriptive only*: it uses ex-post NBER dates and feeds no parameter.

In [ ]:
peaks = usrec.index[(usrec == 0) & (usrec.shift(-1) == 1)]
peaks = [p for p in peaks if p >= macro_raw['FEDFUNDS'].first_valid_index() + 12]
K = np.arange(-36, 37)
spx_lvl_m = (1 + mret['SPX']).cumprod()
panels = {'Unemployment rate, change vs peak (pp)': lambda p, ix: macro_raw['UNRATE'].reindex(ix).values - macro_raw['UNRATE'][p],
          'Fed funds, change vs peak (pp)': lambda p, ix: macro_raw['FEDFUNDS'].reindex(ix).values - macro_raw['FEDFUNDS'][p],
          'S&P 500 total return vs peak (%)': lambda p, ix: 100 * (spx_lvl_m.reindex(ix).values / spx_lvl_m[p] - 1)}
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
for ax, (title, fn), col in zip(axes, panels.items(), [C['red'], C['blue'], C['ink']]):
    paths = np.array([fn(p, [p + int(k) for k in K]) for p in peaks], dtype=float)
    for row in paths:
        ax.plot(K, row, color=C['light'], lw=0.8)
    ax.plot(K, np.nanmedian(paths, axis=0), color=col, lw=2.4, label='median')
    ax.axvline(0, color=C['ink2'], lw=0.8)
    ax.axhline(0, color=C['ink2'], lw=0.6)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('months from NBER peak')
axes[0].legend(loc='upper left')
fig.suptitle(f"{len(peaks)} recessions: {', '.join(str(p.year) for p in peaks)}", x=0.01, ha='left', fontsize=9,
             color=C['ink2'])
fig.tight_layout()
plt.show()

## 5 · In-sample market study: how equities behave in each phase
Window: first phase observation to **`DESIGN_END` (2004-12)**, so nothing from the 2005+ test set is used. Returns are those earned in the month *after* each phase reading. S&P 500 history before 1988 is the price index plus the dividend approximation; Nasdaq-100 history starts in 1985.

In [ ]:
def fwd_return(r, k):
    """Compounded return over the k months after each month-end."""
    return np.exp(np.log1p(r).rolling(k).sum().shift(-k)) - 1


def phase_study(ph, sleeve, end=DESIGN_END):
    df = pd.DataFrame({'phase': ph, 'r': mret[sleeve].shift(-1), 'c': mret['CASH'].shift(-1),
                       'f6': fwd_return(mret[sleeve], 6), 'f12': fwd_return(mret[sleeve], 12)}).loc[:end]
    df = df.dropna(subset=['phase', 'r', 'c'])
    run_len = pd.Series({p: np.mean([(b - a).n + 1 for a, b, v in spells(ph.loc[:end]) if v == p] or [np.nan])
                         for p in PHASES})
    rows = {}
    for p, g in df.groupby('phase'):
        ex = g['r'] - g['c']
        rows[p] = {'months': len(g), 'avg spell (m)': run_len[p],
                   'ann. return': (1 + g['r']).prod() ** (12 / len(g)) - 1,
                   'ann. vol': g['r'].std() * np.sqrt(12),
                   'Sharpe': ex.mean() / ex.std() * np.sqrt(12) if len(g) > 2 else np.nan,
                   'hit rate vs cash': (ex > 0).mean(), 'worst month': g['r'].min(),
                   'avg fwd 6m': g['f6'].mean(), 'avg fwd 12m': g['f12'].mean()}
    return pd.DataFrame(rows).T.reindex(PHASES)


STUDY_FMT = {'months': '{:.0f}', 'avg spell (m)': '{:.1f}', 'ann. return': '{:.1%}', 'ann. vol': '{:.1%}',
             'Sharpe': '{:.2f}', 'hit rate vs cash': '{:.0%}', 'worst month': '{:.1%}',
             'avg fwd 6m': '{:.1%}', 'avg fwd 12m': '{:.1%}'}
study_spx = phase_study(phase, 'SPX')
study_ndx = phase_study(phase.loc[BACKTEST_START:], 'NDX')
print(f'S&P 500, {phase.index[0]} to {DESIGN_END}')
display(study_spx.style.format(STUDY_FMT, na_rep='-'))
print(f'Nasdaq-100, {BACKTEST_START} to {DESIGN_END}')
display(study_ndx.style.format(STUDY_FMT, na_rep='-'))

trans = pd.crosstab(phase.loc[:DESIGN_END], phase.loc[:DESIGN_END].shift(-1), normalize='index')
trans = trans.reindex(index=PHASES, columns=PHASES).fillna(0)
print('Monthly phase transition probabilities (row = this month, column = next month)')
display(trans.style.format('{:.1%}').background_gradient(cmap='Blues', axis=None))

## 6 · Confidence model (walk-forward)
**Target: contraction, as the thesis defines it.** A contraction runs from the NBER business-cycle peak until unemployment reaches its cycle high. The label is `y = 1` if any of the next `HORIZON_M` months falls inside a contraction.

Plain NBER recessions make a poor label for this thesis. The official trough comes *before* the unemployment peak, by up to 19 months in 2001-03. That leaves many months where unemployment is still rising labeled "no recession", so the model would learn the opposite of the thesis.

**Confidence** is `1 - P(y = 1)`, the probability that no contraction is under way or starting in the next six months.

**Model.** L2-regularized logistic regression on the seven cycle features plus phase dummies. Inputs are standardized on the training window and clipped at ±3σ. It is re-fit every 12 months. At each refit the labels are **rebuilt as of that date**: NBER dating is known only up to 12 months earlier, and the unemployment high is the highest reading published so far. A label is used only once its whole horizon falls inside that known window. By default the training window expands, so every month since the 1950s counts equally. `TRAIN_WINDOW_M` and `TRAIN_HALF_LIFE_M` switch to a rolling window or time-decay weights; §11a tests whether that helps.

**Sign constraints from the thesis.** A linear model can still extrapolate oddly in extreme states such as the 2020 unemployment spike. Each coefficient is therefore restricted to the direction the thesis implies; a constrained coefficient can be zero but can never flip:

| Input | Allowed sign of effect on contraction odds |
|---|---|
| Unemployment 6m change, Sahm gap | ≥ 0 (rising unemployment raises risk) |
| Unemployment vs 6m high | ≥ 0 (rolling over from the high lowers risk) |
| Fed funds 6m change, fed funds vs 10q high | ≤ 0 (cuts / a collapse raise risk) |
| Fed funds position in 10q range | ≥ 0 (policy at the top of the range = late cycle) |
| Phase = Peak, Recession | ≥ 0; Phase = Recovery ≤ 0 (vs Expansion) |
| Unemployment position in 10q range | free |

In [ ]:
FEATURES = ['ur_trend', 'ur_gap', 'ur_off_high', 'ur_pos', 'ff_trend', 'ff_pos', 'ff_off_peak']
PHASE_DUMMIES = ['Peak', 'Recession', 'Recovery']                 # Expansion is the baseline
FEATURE_LABELS = {
    'ur_trend': 'Unemployment 6m change', 'ur_gap': 'Unemployment vs 12m low (Sahm gap)',
    'ur_off_high': 'Unemployment vs 6m high', 'ur_pos': f'Unemployment position in {LOOKBACK_Q}q range',
    'ff_trend': 'Fed funds 6m change', 'ff_pos': f'Fed funds position in {LOOKBACK_Q}q range',
    'ff_off_peak': f'Fed funds vs {LOOKBACK_Q}q high', 'is_peak': 'Phase = Peak',
    'is_recession': 'Phase = Recession', 'is_recovery': 'Phase = Recovery'}


def design(X, ph):
    D = X[FEATURES].copy()
    for p in PHASE_DUMMIES:
        D['is_' + p.lower()] = (ph == p).astype(float)
    return D


def contraction_flag(macro, ur_peak_window=24):
    """1 from each NBER peak until unemployment's cycle high, searched up to `ur_peak_window` months after the
    trough but never past the start of the next recession. On truncated data an unfinished recession runs to
    the highest unemployment reading so far."""
    rec, ur = macro['USREC'].dropna(), macro['UNRATE'].dropna()
    flag = pd.Series(0.0, index=rec.index)
    starts = list(rec.index[(rec == 1) & (rec.shift(1) == 0)])
    for start, nxt in zip(starts, starts[1:] + [None]):
        after = rec.loc[start:]
        trough = after[after == 0].index[0] - 1 if (after == 0).any() else after.index[-1]
        end = trough + ur_peak_window if nxt is None else min(trough + ur_peak_window, nxt - 1)
        window = ur.loc[start:end]
        if len(window):
            flag.loc[start:window.idxmax()] = 1.0
    return flag


def event_ahead(flag, h):
    """1 if any flagged month falls in (t, t+h]; NaN where the window runs past the data."""
    fwd = pd.concat([flag.shift(-k) for k in range(1, h + 1)], axis=1)
    y = fwd.max(axis=1)
    y[fwd.isna().any(axis=1)] = np.nan
    return y


def make_label_fn(macro, h=HORIZON_M, lag=NBER_LAG_M):
    """label_fn(asof) -> training labels rebuilt from ONLY the data published by decision month `asof`:
    unemployment up to asof-1, NBER recession dating up to asof-lag. No label ever uses hindsight."""
    def label_fn(asof):
        m = macro.loc[:asof - PUB_LAG_M].copy()
        m.loc[m.index > asof - lag, 'USREC'] = np.nan
        y_ = event_ahead(contraction_flag(m), h)
        return y_[y_.index <= asof - (h + lag)].dropna()
    return label_fn


SIGNS = {'ur_trend': 1, 'ur_gap': 1, 'ur_off_high': 1, 'ur_pos': 0, 'ff_trend': -1, 'ff_pos': 1, 'ff_off_peak': -1,
         'is_peak': 1, 'is_recession': 1, 'is_recovery': -1}      # +1 / -1 = coefficient sign, 0 = free


class SignedLogit:
    """L2-regularized logistic regression with sign constraints (L-BFGS-B) on clipped z-scored inputs."""

    def __init__(self, signs=SIGNS, C=L2_C, clip=Z_CLIP):
        self.signs, self.C, self.clip = signs, C, clip

    def _z(self, X):
        return ((X[self.cols] - self.mu) / self.sd).clip(-self.clip, self.clip).to_numpy(float)

    def fit(self, X, y, sample_weight=None, scale_ref=None):
        """`sample_weight`: optional per-row weights (time decay). `scale_ref`: frame whose mean/std define the
        z-scores (so coefficients from different sub-samples are comparable); default = the training rows."""
        self.cols = list(X.columns)
        sw = np.ones(len(y)) if sample_weight is None else np.asarray(sample_weight, float)
        sw = sw / sw.mean()
        ref, rw = (X, sw) if scale_ref is None else (scale_ref[self.cols], np.ones(len(scale_ref)))
        self.mu = pd.Series(np.average(ref, axis=0, weights=rw), index=self.cols)
        self.sd = pd.Series(np.sqrt(np.average((ref - self.mu) ** 2, axis=0, weights=rw)), index=self.cols)
        self.sd = self.sd.replace(0, 1.0)
        Z, yv = self._z(X), y.to_numpy(float)

        def loss(w):
            eta = w[0] + Z @ w[1:]
            g = sw * (expit(eta) - yv)
            return (sw * (np.logaddexp(0, eta) - yv * eta)).sum() + 0.5 / self.C * w[1:] @ w[1:], \
                np.r_[g.sum(), Z.T @ g + w[1:] / self.C]

        sign = [self.signs.get(c, 0) for c in self.cols]
        bounds = [(None, None)] + [(0, None) if s > 0 else (None, 0) if s < 0 else (None, None) for s in sign]
        w = minimize(loss, np.zeros(len(self.cols) + 1), jac=True, method='L-BFGS-B', bounds=bounds).x
        self.intercept_, self.coef_ = w[0], pd.Series(w[1:], index=self.cols)
        return self

    def predict_proba(self, X):
        p = expit(self.intercept_ + self._z(X) @ self.coef_.to_numpy())
        return np.c_[1 - p, p]


def fit_model(D, y, w=None):
    return SignedLogit().fit(D, y, w)


def trainable(D, label_fn, asof, window_m=None, half_life_m=None):
    """Design rows and as-of labels knowable at decision month `asof`, with an optional rolling window
    (`window_m` months) or exponential time-decay weights (`half_life_m` months)."""
    yy = label_fn(asof)
    idx = yy.index.intersection(D.index)
    age = np.asarray((asof.year * 12 + asof.month) - (idx.year * 12 + idx.month))
    keep = age < window_m if window_m else np.ones(len(idx), bool)
    w = 0.5 ** (age[keep] / half_life_m) if half_life_m else np.ones(keep.sum())
    return D.loc[idx[keep]], yy.loc[idx[keep]], w


def walk_forward(D, label_fn, start=BACKTEST_START, every=REFIT_EVERY_M, window_m=TRAIN_WINDOW_M,
                 half_life_m=TRAIN_HALF_LIFE_M, min_obs=120):
    months = D.index[D.index >= pd.Period(start, 'M')]
    p = pd.Series(np.nan, index=months, name='p_contraction')
    coefs = {}
    for i in range(0, len(months), every):
        block = months[i:i + every]
        Xt, yt, w = trainable(D, label_fn, block[0], window_m, half_life_m)
        if len(yt) < min_obs or yt.nunique() < 2:
            continue
        model = fit_model(Xt, yt, w)
        p.loc[block] = model.predict_proba(D.loc[block])[:, 1]
        coefs[block[0]] = model.coef_
    return p, pd.DataFrame(coefs).T


def step(c, s=CONF_STEP):
    return (np.round(c / s) * s).round(4).clip(0, 1)


def confidence_pipeline(macro, lookback_m=LOOKBACK_M, h=HORIZON_M, start=BACKTEST_START, **train):
    X_ = to_decision_time(build_features(macro, lookback_m))
    ph = classify_phase(X_)
    D_ = design(X_, ph)
    label_fn = make_label_fn(macro, h)
    p, coefs = walk_forward(D_, label_fn, start, **train)
    y_final = event_ahead(contraction_flag(macro), h)       # hindsight labels: used ONLY to score the forecasts
    return dict(X=X_, phase=ph, D=D_, y=y_final, label_fn=label_fn, p=p, coefs=coefs, conf=1 - p,
                conf_step=step(1 - p))


pipe = confidence_pipeline(macro_raw)
D, y, p_risk, conf, conf_step = pipe['D'], pipe['y'], pipe['p'], pipe['conf'], pipe['conf_step']
contraction = contraction_flag(macro_raw)
print('Contractions (NBER peak -> unemployment high):',
      ', '.join(f'{a}..{b}' for a, b, v in spells(contraction) if v == 1 and b >= X.index[0]))
conf_step = conf_step.dropna()
print(f'Out-of-sample confidence: {conf_step.index[0]} -> {conf_step.index[-1]}, '
      f'{len(pipe["coefs"])} walk-forward fits')

# --- diagnostics (scored against final, hindsight labels: evaluation only, never used for fitting) ---
ev = pd.DataFrame({'p': p_risk, 'y': y}).dropna()
base = pd.Series(ev['y'].expanding().mean().shift(1).fillna(ev['y'].iloc[0]))
print(f"AUC {roc_auc_score(ev.y, ev.p):.3f} | Brier {brier_score_loss(ev.y, ev.p):.4f} "
      f"(naive base rate {brier_score_loss(ev.y, base):.4f}) | contraction share {ev.y.mean():.1%}")
calib = ev.groupby(pd.cut(ev.p, [0, .1, .2, .4, .6, .8, 1.0], include_lowest=True), observed=True).agg(
    months=('y', 'size'), predicted=('p', 'mean'), realized=('y', 'mean'))
display(calib.style.format({'months': '{:.0f}', 'predicted': '{:.1%}', 'realized': '{:.1%}'}))

# --- interpretation: full-sample fit (NOT used in the backtest) + walk-forward sign stability ---
Xf, yf_, _ = trainable(D, pipe['label_fn'], D.index[-1])
full = fit_model(Xf, yf_)
coef = full.coef_
wf_lo, wf_hi = pipe['coefs'].min(), pipe['coefs'].max()
fig, ax = plt.subplots(figsize=(11, 3.8))
order = coef.sort_values().index
ax.barh([FEATURE_LABELS[c] for c in order], coef[order],
        color=[C['red'] if v > 0 else C['blue'] for v in coef[order]], height=0.6)
ax.axvline(0, color=C['ink2'], lw=0.8)
ax.set_title('Standardized coefficients, full-sample fit (red raises contraction odds, blue lowers them)')
ax.text(1.02, len(order) - 0.4, 'walk-forward range', transform=ax.get_yaxis_transform(), fontsize=8,
        color=C['ink2'], fontweight='bold')
for i, c in enumerate(order):
    ax.text(1.02, i, f'{wf_lo[c]:+.2f} to {wf_hi[c]:+.2f}', transform=ax.get_yaxis_transform(), va='center',
            fontsize=8, color=C['ink2'])
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True, gridspec_kw=dict(height_ratios=[2, 1.4]))
ax = axes[0]
shade_nber(ax, conf_step.index[0])
lv = (1 + mret['SPX'].loc[conf_step.index[0]:]).cumprod()
ax.semilogy(lv.index.to_timestamp(), lv, color=C['ink'], lw=1.3)
ax.set_title('S&P 500 total return (log)')
ax = axes[1]
shade(ax, contraction.loc[conf_step.index[0]:], {1: C['red']}, alpha=0.12)
ax.plot(conf.index.to_timestamp(), conf, color=C['muted'], lw=0.9, label='raw')
ax.step(conf_step.index.to_timestamp(), conf_step, where='post', color=C['blue'], lw=1.6, label='10 % steps')
for lvl_, lab in [(USER_TIERS['hi'], 'NDX tier'), (USER_TIERS['lo'], 'SPX tier')]:
    ax.axhline(lvl_, color=C['ink2'], lw=0.7)
    ax.annotate(lab, (conf_step.index[0].start_time, lvl_), xytext=(2, 3), textcoords='offset points', fontsize=8,
                color=C['ink2'])
ax.set_ylim(0, 1.02)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.0%}'))
ax.set_title('Out-of-sample expansion confidence (walk-forward); red = contraction, NBER peak to unemployment high')
ax.legend(loc='lower left', ncol=2)
fig.tight_layout()
plt.show()

### How the market reacted to each confidence level (design window only)
This is the mapping the optimizer exploits. It shows the annualized next-month excess return over cash at each stepped confidence level, using out-of-sample confidence from 1986 to 2004.

In [ ]:
def confidence_buckets(cs, end=DESIGN_END):
    df = pd.DataFrame({'conf': cs, 'spx': mret['SPX'].shift(-1) - mret['CASH'].shift(-1),
                       'ndx': mret['NDX'].shift(-1) - mret['CASH'].shift(-1)}).loc[:end].dropna()
    return df.groupby('conf').agg(months=('spx', 'size'), spx_excess=('spx', 'mean'), ndx_excess=('ndx', 'mean'),
                                  spx_hit=('spx', lambda s: (s > 0).mean()), ndx_hit=('ndx', lambda s: (s > 0).mean()))


buckets = confidence_buckets(conf_step)
buckets[['spx_excess', 'ndx_excess']] *= 12
display(buckets.rename(index=lambda v: f'{v:.0%}').style.format(
    {'months': '{:.0f}', 'spx_excess': '{:.1%}', 'ndx_excess': '{:.1%}', 'spx_hit': '{:.0%}', 'ndx_hit': '{:.0%}'}))
fig, ax = plt.subplots(figsize=(10, 3.4))
xx = np.arange(len(buckets))
ax.bar(xx - 0.2, buckets['spx_excess'], 0.38, color=C['blue'], label='S&P 500')
ax.bar(xx + 0.2, buckets['ndx_excess'], 0.38, color=C['orange'], label='Nasdaq-100')
ax.set_xticks(xx, [f'{v:.0%}\n(n={n})' for v, n in zip(buckets.index, buckets['months'])])
ax.axhline(0, color=C['ink2'], lw=0.8)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.0%}'))
ax.set_title(f'Annualized next-month excess return by stepped confidence ({BACKTEST_START} to {DESIGN_END})')
ax.legend()
plt.tight_layout()
plt.show()

## 7 · Allocation rules and backtest engine
* **Proportional 50/50** (first idea): equity weight equals stepped confidence, split evenly between S&P 500 and Nasdaq-100, with the rest in cash. At 90 % confidence that is 45 % S&P, 45 % NDX and 10 % cash.
* **Tiered** (second idea): confidence ≥ 70 % → 100 % Nasdaq-100; 40-69 % → 100 % S&P 500; below 40 % → cash.
* **Engine.**
  * The decision for month *M* is taken after *M*'s last close and filled at the close `EXEC_LAG_DAYS` later.
  * Holdings drift with prices until the next fill.
  * Each fill pays `COST_BPS` per unit of turnover in the equity sleeves.
  * Cash earns the prior month's 3-month T-bill yield.

In [ ]:
def alloc_proportional(cs):
    c = cs.to_numpy()
    return pd.DataFrame({'SPX': c / 2, 'NDX': c / 2, 'CASH': 1 - c}, index=cs.index)


def alloc_tiered(cs, hi, lo, top, mid, low):
    c = cs.to_numpy()
    band = np.select([c >= hi - 1e-9, c >= lo - 1e-9], [0, 1], 2)
    table = np.array([top, mid, low], dtype=float)
    spx, ndx = table[band, 0], table[band, 1]
    return pd.DataFrame({'SPX': spx, 'NDX': ndx, 'CASH': 1 - spx - ndx}, index=cs.index)


def alloc_constant(index, spx, ndx):
    return pd.DataFrame({'SPX': spx, 'NDX': ndx, 'CASH': 1 - spx - ndx}, index=index)


def band_text(t, step_=CONF_STEP):
    """Human-readable tier rule, e.g. '>= 70%: 100% Nasdaq-100 | 40-60%: ...'."""
    mid = f"{t['lo']:.0%}" if t['hi'] - step_ <= t['lo'] + 1e-9 else f"{t['lo']:.0%}-{t['hi'] - step_:.0%}"
    return (f">= {t['hi']:.0%}: {describe_alloc(*t['top'])} | {mid}: {describe_alloc(*t['mid'])} | "
            f"< {t['lo']:.0%}: {describe_alloc(*t['low'])}")


def describe_alloc(spx, ndx):
    parts = [f'{v:.0%} {k}' for k, v in [('S&P 500', spx), ('Nasdaq-100', ndx), ('Cash', 1 - spx - ndx)] if v > 1e-9]
    return ' / '.join(parts)


def run_backtest(weights, cost_bps=COST_BPS, exec_lag=EXEC_LAG_DAYS, end=None, R=None):
    """Drift-aware monthly-rebalanced backtest. Returns daily net returns, end-of-day weights and turnover."""
    R = rets if R is None else R
    R = R if end is None else R.loc[:pd.Period(end, 'M').end_time]
    days = R.index
    last_day = pd.Series(np.arange(len(days)), index=days).groupby(days.to_period('M')).max()
    W = weights[ASSETS].dropna()
    W = W[W.index.isin(last_day.index)]
    fill = last_day.loc[W.index].to_numpy() + exec_lag
    ok = fill < len(days)
    W, fill = W[ok], fill[ok]
    if len(W) == 0:
        raise ValueError('no fills inside the return window')
    assert not R.iloc[fill[0] + 1:][W.columns[(W > 0).any()]].isna().any().any(), 'weight on a sleeve without data'
    period = np.searchsorted(fill, np.arange(len(days)), side='left') - 1    # last fill strictly before each day
    first = fill[0] + 1
    per = period[first:]
    gross = pd.DataFrame(1 + R.iloc[first:].fillna(0).to_numpy(), columns=ASSETS)
    G = gross.groupby(per).cumprod().to_numpy()
    hold = W.to_numpy()[per] * G                                # sleeve value relative to NAV at the last fill
    V = hold.sum(1)
    r = V / pd.Series(V).groupby(per).shift(1).fillna(1.0).to_numpy() - 1
    w_eod = hold / V[:, None]
    pre = np.vstack([[0.0, 0.0, 1.0], w_eod[fill[1:] - first]])    # weights just before each fill (start in cash)
    turnover = np.abs(W.to_numpy()[:, :2] - pre[:, :2]).sum(1)
    cost = np.zeros(len(r))
    np.add.at(cost, np.r_[0, fill[1:] - first], turnover * cost_bps / 1e4)
    idx = days[first:]
    return dict(ret=pd.Series((1 + r) * (1 - cost) - 1, index=idx),
                w=pd.DataFrame(w_eod, index=idx, columns=ASSETS),
                turnover=pd.Series(turnover, index=days[fill]), target=W,
                decision=pd.Series(W.index[per], index=idx))          # decision month driving each day


def perf_stats(r, w=None, turnover=None, bench=None):
    r = r.dropna()
    rf = rets['CASH'].reindex(r.index).fillna(0)
    ex = r - rf
    yrs = len(r) / 252
    nav = (1 + r).cumprod()
    cagr = nav.iloc[-1] ** (1 / yrs) - 1
    dvol = np.sqrt((np.minimum(ex, 0) ** 2).mean()) * np.sqrt(252)
    dd = nav / nav.cummax() - 1
    uw = dd < 0
    mp = r.index.to_period('M')
    m = (1 + r).groupby(mp).prod() - 1
    mex = m - ((1 + rf).groupby(mp).prod() - 1)
    srm, sk, ku = mex.mean() / mex.std(), stats.skew(mex), stats.kurtosis(mex, fisher=False)
    var95 = r.quantile(0.05)
    out = {'CAGR': cagr, 'Total Return': nav.iloc[-1] - 1, 'Ann. Volatility': r.std() * np.sqrt(252),
           'Downside Vol': dvol, 'Sharpe': ex.mean() / ex.std() * np.sqrt(252), 'Sortino': ex.mean() * 252 / dvol,
           'Max Drawdown': dd.min(), 'Calmar': cagr / abs(dd.min()),
           'Max DD Duration (yrs)': uw.groupby((~uw).cumsum()).sum().max() / 252,
           'Ulcer Index': np.sqrt((dd ** 2).mean()), 'VaR 95% (daily)': var95,
           'CVaR 95% (daily)': r[r <= var95].mean(), 'Best Month': m.max(), 'Worst Month': m.min(),
           '% Positive Months': (m > 0).mean(), 'Skew (monthly)': sk, 'Kurtosis (monthly)': ku,
           'PSR (SR>0)': stats.norm.cdf(srm * np.sqrt(len(mex) - 1) / np.sqrt(1 - sk * srm + (ku - 1) / 4 * srm ** 2))}
    if bench is not None:
        bm = (1 + bench.reindex(r.index).fillna(0)).groupby(mp).prod() - 1
        bex = bm - (m - mex)
        beta = np.cov(mex, bex)[0, 1] / bex.var()
        act = m - bm
        te = act.std() * np.sqrt(12)
        out.update({'Beta (vs S&P)': beta, 'Alpha (ann.)': (mex.mean() - beta * bex.mean()) * 12,
                    'Correlation': np.corrcoef(m, bm)[0, 1], 'Tracking Error': te,
                    'Information Ratio': act.mean() * 12 / te if te > 1e-9 else np.nan,
                    'Up Capture': m[bm > 0].mean() / bm[bm > 0].mean(),
                    'Down Capture': m[bm < 0].mean() / bm[bm < 0].mean()})
    if w is not None:
        ww = w.reindex(r.index)
        eq = ww['SPX'] + ww['NDX']
        out.update({'Avg Equity Exposure': eq.mean(), 'Avg NDX Weight': ww['NDX'].mean(),
                    'Time Invested': (eq > 0.01).mean()})
    if turnover is not None:
        out['Turnover (ann.)'] = turnover.loc[r.index[0]:r.index[-1]].sum() / yrs
    return pd.Series(out)


def monthly_excess(r):
    mp = r.index.to_period('M')
    return (1 + r).groupby(mp).prod() - (1 + rets['CASH'].reindex(r.index)).groupby(mp).prod()


def sharpe(r):
    ex = r - rets['CASH'].reindex(r.index).fillna(0)
    return ex.mean() / ex.std() * np.sqrt(252)


PCT_ROWS = {'CAGR', 'Total Return', 'Ann. Volatility', 'Downside Vol', 'Max Drawdown', 'Ulcer Index',
            'VaR 95% (daily)', 'CVaR 95% (daily)', 'Best Month', 'Worst Month', '% Positive Months', 'PSR (SR>0)',
            'Alpha (ann.)', 'Tracking Error', 'Up Capture', 'Down Capture', 'Avg Equity Exposure', 'Avg NDX Weight',
            'Time Invested', 'Turnover (ann.)'}


def fmt_table(df):
    out = df.astype(object).copy()
    for row in df.index:
        f = '{:.1%}' if row in PCT_ROWS else '{:.2f}'
        out.loc[row] = [f.format(v) if pd.notna(v) else '-' for v in df.loc[row]]
    return out


def stats_table(results, start=None, end=None):
    bench = results['S&P 500 B&H']['ret']
    return pd.DataFrame({k: perf_stats(v['ret'].loc[start:end], v['w'], v['turnover'], bench)
                         for k, v in results.items()})


# sanity check: a 100 % S&P allocation must reproduce the S&P sleeve exactly (no costs after the first fill)
_chk = run_backtest(alloc_constant(conf_step.index, 1.0, 0.0), cost_bps=0)
assert np.allclose(_chk['ret'], rets['SPX'].loc[_chk['ret'].index]), 'engine check failed'
print('Engine check passed: constant 100 % S&P allocation reproduces the S&P sleeve.')

## 8 · Strategies, optimization and benchmarks
**Optimizer.** It searches tier thresholds `hi` and `lo` in 10 % steps, plus the allocation of each band:
* top band: 100 % NDX, 50/50, or 100 % SPX
* middle band: 100 % SPX, 50/50, or 50 % SPX + 50 % cash
* low band: cash, or 50 % SPX + 50 % cash

It maximizes `OPT_OBJECTIVE` on **1986-2004 only** (`BACKTEST_START` to `DESIGN_END`). To avoid knife-edge optima, it picks the configuration with the best *neighborhood-average* objective, averaged over adjacent `hi`/`lo` cells, rather than the single best cell.

**Extension (not in the original spec).** *Optimized + Trend* keeps the optimized tiers, but when the S&P 500 closes the month above its 10-month SMA, confidence is floored at the middle band. This targets the thesis's known weak spot: unemployment peaks *after* the market bottoms, so a pure macro signal re-enters late.

In [ ]:
dec = conf_step.loc[BACKTEST_START:]
dec_design = dec.loc[:DESIGN_END]


BANDS = {'top': {'100% NDX': (0.0, 1.0), '50/50': (0.5, 0.5), '100% SPX': (1.0, 0.0)},
         'mid': {'100% SPX': (1.0, 0.0), '50/50': (0.5, 0.5), '50% SPX': (0.5, 0.0)},
         'low': {'Cash': (0.0, 0.0), '50% SPX': (0.5, 0.0)}}
HI_GRID = np.round(np.arange(0.5, 0.95, 0.1), 2)
LO_GRID = np.round(np.arange(0.1, 0.85, 0.1), 2)

rows, grid_m = [], []
for tn, top in BANDS['top'].items():
    for mn, mid in BANDS['mid'].items():
        for ln, low in BANDS['low'].items():
            for hi in HI_GRID:
                for lo in LO_GRID[LO_GRID < hi - 1e-9]:
                    r_cfg = run_backtest(alloc_tiered(dec_design, hi, lo, top, mid, low), end=DESIGN_END)['ret']
                    s = perf_stats(r_cfg)
                    grid_m.append(monthly_excess(r_cfg).to_numpy())
                    rows.append(dict(top=tn, mid=mn, low=ln, hi=hi, lo=lo, Sharpe=s['Sharpe'], CAGR=s['CAGR'],
                                     Sortino=s['Sortino'], Calmar=s['Calmar'], MaxDD=s['Max Drawdown']))
grid = pd.DataFrame(rows)


def neighborhood(g, col):
    """Average of `col` over each (hi, lo) cell and its valid neighbors in the threshold grid."""
    piv = g.pivot(index='hi', columns='lo', values=col)
    vals = piv.to_numpy()
    pos = {(h, l): (i, j) for i, h in enumerate(piv.index) for j, l in enumerate(piv.columns)}
    return [np.nanmean(vals[max(i - 1, 0):i + 2, max(j - 1, 0):j + 2])
            for i, j in (pos[(h, l)] for h, l in zip(g['hi'], g['lo']))]


grid['smoothed'] = np.nan
for _, g in grid.groupby(['top', 'mid', 'low']):
    grid.loc[g.index, 'smoothed'] = neighborhood(g, OPT_OBJECTIVE)
best = grid.loc[grid['smoothed'].idxmax()]
BEST = dict(hi=float(best.hi), lo=float(best.lo), top=BANDS['top'][best.top], mid=BANDS['mid'][best.mid],
            low=BANDS['low'][best.low])
print(f"{len(grid)} configurations evaluated on {BACKTEST_START} to {DESIGN_END}")
print('Selected rule:', band_text(BEST))
print(f"  design-period {OPT_OBJECTIVE}: {best[OPT_OBJECTIVE]:.2f} (neighborhood avg {best.smoothed:.2f})")
display(grid.sort_values('smoothed', ascending=False).head(10).style.format(
    {'Sharpe': '{:.2f}', 'CAGR': '{:.1%}', 'Sortino': '{:.2f}', 'Calmar': '{:.2f}', 'MaxDD': '{:.1%}',
     'smoothed': '{:.2f}', 'hi': '{:.0%}', 'lo': '{:.0%}'}))

sel = grid[(grid.top == best.top) & (grid.mid == best.mid) & (grid.low == best.low)]
hm = sel.pivot(index='hi', columns='lo', values=OPT_OBJECTIVE)
fig, ax = plt.subplots(figsize=(7.5, 3.6))
im = ax.imshow(hm.to_numpy(), cmap='Blues', aspect='auto', origin='lower')
ax.set_xticks(range(hm.shape[1]), [f'{v:.0%}' for v in hm.columns])
ax.set_yticks(range(hm.shape[0]), [f'{v:.0%}' for v in hm.index])
ax.set_xlabel('lo threshold')
ax.set_ylabel('hi threshold')
ax.grid(False)
for (i, j), v in np.ndenumerate(hm.to_numpy()):
    if not np.isnan(v):
        ax.text(j, i, f'{v:.2f}', ha='center', va='center', fontsize=8,
                color='white' if v > np.nanpercentile(hm, 70) else C['ink'])
ax.set_title(f'Design-period {OPT_OBJECTIVE}: top={best.top}, mid={best.mid}, low={best.low}')
fig.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()

# Deflated Sharpe Ratio (Bailey & Lopez de Prado): is the selected design-period Sharpe still significant
# after trying len(grid) configurations?
sel_ret = run_backtest(alloc_tiered(dec_design, **BEST), end=DESIGN_END)['ret']
mp_ = sel_ret.index.to_period('M')
mex_ = (1 + sel_ret).groupby(mp_).prod() - (1 + rets['CASH'].reindex(sel_ret.index)).groupby(mp_).prod()
sr_m, sk_, ku_ = mex_.mean() / mex_.std(), stats.skew(mex_), stats.kurtosis(mex_, fisher=False)
var_sr = (grid['Sharpe'] / np.sqrt(12)).var()
euler = 0.5772156649
sr0 = np.sqrt(var_sr) * ((1 - euler) * stats.norm.ppf(1 - 1 / len(grid)) + euler * stats.norm.ppf(1 - 1 / (len(grid) * np.e)))
dsr = stats.norm.cdf((sr_m - sr0) * np.sqrt(len(mex_) - 1) / np.sqrt(1 - sk_ * sr_m + (ku_ - 1) / 4 * sr_m ** 2))
print(f'Deflated Sharpe Ratio: {dsr:.1%} probability the selected rule has true Sharpe > 0 after '
      f'{len(grid)} trials (expected max Sharpe by luck: {sr0 * np.sqrt(12):.2f} ann.)')

In [ ]:
trend_up = (spx_px_m > spx_px_m.rolling(TREND_SMA_M).mean()).reindex(dec.index).fillna(False).astype(bool)
dec_trend = dec.where(~trend_up, np.maximum(dec, BEST['lo']))

STRATS = {
    'Proportional 50/50': alloc_proportional(dec),
    'Tiered 70/40 (user)': alloc_tiered(dec, **USER_TIERS),
    'Optimized Tiers': alloc_tiered(dec, **BEST),
    'Optimized + Trend': alloc_tiered(dec_trend, **BEST),
}
avg_w = STRATS['Optimized Tiers'].mean()
BENCH = {
    'S&P 500 B&H': alloc_constant(dec.index, 1.0, 0.0),
    'Nasdaq-100 B&H': alloc_constant(dec.index, 0.0, 1.0),
    '50/50 SPX/NDX': alloc_constant(dec.index, 0.5, 0.5),
    'S&P 10m-SMA Trend': alloc_constant(dec.index, trend_up.astype(float).to_numpy(), 0.0),
    'Exposure-Matched Static': alloc_constant(dec.index, avg_w['SPX'], avg_w['NDX']),
}
results = {k: run_backtest(v) for k, v in {**BENCH, **STRATS}.items()}
STRAT_NAMES, BENCH_NAMES = list(STRATS), list(BENCH)
STYLE = {'Proportional 50/50': dict(color=C['blue']), 'Tiered 70/40 (user)': dict(color=C['orange']),
         'Optimized Tiers': dict(color=C['aqua']), 'Optimized + Trend': dict(color=C['yellow']),
         'S&P 500 B&H': dict(color=C['ink2']), 'Nasdaq-100 B&H': dict(color=C['light']),
         '50/50 SPX/NDX': dict(color=C['muted'], ls='--'), 'S&P 10m-SMA Trend': dict(color=C['muted'], ls=':'),
         'Exposure-Matched Static': dict(color=C['light'], ls='-.')}
print(f"Exposure-matched static mix: {describe_alloc(avg_w['SPX'], avg_w['NDX'])}")
first_day = results['S&P 500 B&H']['ret'].index[0].date()
print(f'Backtest: {first_day} -> {rets.index[-1].date()}')

## 9 · Results
The full-period numbers mix the design window, where the optimized rule was fitted, with the test window. The **2005-today test window** is the honest out-of-sample read for *Optimized Tiers* and *Optimized + Trend*. The user's two rules and the benchmarks have no fitted parameters.

In [ ]:
HEADLINE = ['CAGR', 'Ann. Volatility', 'Sharpe', 'Sortino', 'Max Drawdown', 'Calmar', 'Avg Equity Exposure',
            'Beta (vs S&P)', 'Up Capture', 'Down Capture']
test_start = (pd.Period(DESIGN_END, 'M') + 1).strftime('%Y-%m')
windows = {'Full period': (None, None), f'Design ({BACKTEST_START} to {DESIGN_END})': (None, DESIGN_END),
           f'Test ({test_start} to today)': (test_start, None)}
tables = {name: stats_table(results, a, b) for name, (a, b) in windows.items()}
for name, t in tables.items():
    print(name)
    display(fmt_table(t.loc[HEADLINE]))

In [ ]:
print('Full-period detail')
display(fmt_table(tables['Full period']))

In [ ]:
def plot_lines(ax, series_by_name, names, log=False, lw_main=1.8):
    for n in names:
        s = series_by_name[n]
        st = dict(STYLE[n])
        st.setdefault('lw', lw_main if n in STRAT_NAMES else 1.2)
        (ax.semilogy if log else ax.plot)(s.index, s, label=n, **st)


show = STRAT_NAMES + ['S&P 500 B&H', 'Nasdaq-100 B&H']
navs = {k: (1 + v['ret']).cumprod() for k, v in results.items()}
dds = {k: v / v.cummax() - 1 for k, v in navs.items()}

fig, axes = plt.subplots(2, 1, figsize=(12, 8.5), sharex=True, gridspec_kw=dict(height_ratios=[2.2, 1]))
shade_nber(axes[0], dec.index[0])
plot_lines(axes[0], navs, show, log=True)
axes[0].set_title('Growth of $1 (log scale, net of costs)')
axes[0].legend(loc='upper left', ncol=3)
shade_nber(axes[1], dec.index[0])
plot_lines(axes[1], dds, show)
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.0%}'))
axes[1].set_title('Drawdown')
fig.tight_layout()
plt.show()

In [ ]:
def rolling_sharpe(r, months=36):
    m = (1 + r).groupby(r.index.to_period('M')).prod() - 1
    c = (1 + rets['CASH'].reindex(r.index)).groupby(r.index.to_period('M')).prod() - 1
    ex = m - c
    out = ex.rolling(months).mean() / ex.rolling(months).std() * np.sqrt(12)
    return out.set_axis(out.index.to_timestamp())


fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
roll = {k: rolling_sharpe(v['ret']) for k, v in results.items()}
plot_lines(axes[0], roll, ['Tiered 70/40 (user)', 'Optimized Tiers', 'Optimized + Trend', 'S&P 500 B&H'])
axes[0].axhline(0, color=C['ink2'], lw=0.7)
axes[0].set_title('Rolling 36-month Sharpe ratio')
axes[0].legend(loc='lower left', ncol=4)
vol = {k: v['ret'].rolling(63).std() * np.sqrt(252) for k, v in results.items()}
plot_lines(axes[1], vol, ['Optimized Tiers', 'Optimized + Trend', 'S&P 500 B&H', 'Nasdaq-100 B&H'], lw_main=1.2)
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.0%}'))
axes[1].set_title('Rolling 3-month annualized volatility')
axes[1].legend(loc='upper left', ncol=4)
fig.tight_layout()
plt.show()

In [ ]:
def plot_exposure(name):
    w = results[name]['w']
    fig, axes = plt.subplots(2, 1, figsize=(12, 5.6), sharex=True, gridspec_kw=dict(height_ratios=[1.6, 1]))
    ax = axes[0]
    ax.stackplot(w.index, w['SPX'], w['NDX'], w['CASH'], colors=[C['blue'], C['orange'], C['grid']],
                 labels=['S&P 500', 'Nasdaq-100', 'Cash'], lw=0)
    ax.set_ylim(0, 1)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.0%}'))
    ax.set_title(f'{name}: portfolio weights')
    ax.legend(loc='upper left', ncol=3)
    ax = axes[1]
    shade_nber(ax, dec.index[0])
    ax.step(dec.index.to_timestamp(), dec, where='post', color=C['ink2'], lw=1.2)
    ax.set_ylim(0, 1.02)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.0%}'))
    ax.set_title('Stepped confidence (grey = NBER recession)')
    fig.tight_layout()
    plt.show()


plot_exposure('Optimized Tiers')

In [ ]:
annual = pd.DataFrame({k: (1 + v['ret']).groupby(v['ret'].index.year).prod() - 1 for k, v in results.items()})
annual = annual[STRAT_NAMES + BENCH_NAMES]
print('Calendar-year returns (first year is partial)')
display(annual.style.format('{:.1%}').background_gradient(cmap='RdBu', vmin=-0.4, vmax=0.4, axis=None))

r_best = results['Optimized Tiers']['ret']
mtab = ((1 + r_best).groupby([r_best.index.year, r_best.index.month]).prod() - 1).unstack()
fig, ax = plt.subplots(figsize=(10, max(4, len(mtab) * 0.18)))
lim = np.nanpercentile(np.abs(mtab.to_numpy()), 98)
im = ax.imshow(mtab.to_numpy(), cmap='RdBu', vmin=-lim, vmax=lim, aspect='auto')
ax.set_xticks(range(12), ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'])
ax.set_yticks(range(len(mtab)), mtab.index)
ax.tick_params(axis='y', labelsize=7)
ax.grid(False)
ax.set_title('Optimized Tiers: monthly returns')
fig.colorbar(im, ax=ax, shrink=0.6, format=plt.FuncFormatter(lambda v, _: f'{v:.0%}'))
plt.tight_layout()
plt.show()

### Performance by phase (backtest window)
Annualized return earned while each phase was the live reading. This shows *where* the regime filter adds or loses value.

In [ ]:
by_phase = {}
for n in STRAT_NAMES + ['S&P 500 B&H', 'Nasdaq-100 B&H']:
    res = results[n]
    lab = pd.Series(phase.reindex(res['decision']).to_numpy(), index=res['ret'].index)   # phase that set the weights
    by_phase[n] = res['ret'].groupby(lab).apply(lambda x: (1 + x).prod() ** (252 / len(x)) - 1)
by_phase = pd.DataFrame(by_phase).reindex(PHASES)
by_phase.insert(0, 'share of days', lab.value_counts(normalize=True).reindex(PHASES))
display(by_phase.style.format('{:.1%}', na_rep='-'))

## 10 · Robustness
### 10a · Lookback window and forecast horizon (fixed user rules, so nothing is re-optimized)

In [ ]:
def sensitivity(lookback_q, h):
    pp = confidence_pipeline(macro_raw, lookback_q * 3, h)
    cs = pp['conf_step'].dropna().loc[BACKTEST_START:]
    ev_ = pd.DataFrame({'p': pp['p'], 'y': pp['y']}).dropna()
    row = {'lookback (q)': lookback_q, 'horizon (m)': h, 'AUC': roc_auc_score(ev_.y, ev_.p)}
    for lab, W_ in [('Tiered', alloc_tiered(cs, **USER_TIERS)), ('Proportional', alloc_proportional(cs))]:
        s = perf_stats(run_backtest(W_)['ret'])
        row.update({f'{lab} Sharpe': s['Sharpe'], f'{lab} CAGR': s['CAGR'], f'{lab} MaxDD': s['Max Drawdown'],
                    f'{lab} Calmar': s['Calmar']})
    return row


sens = pd.DataFrame([sensitivity(q, HORIZON_M) for q in [6, 8, 10, 12, 14]]
                    + [sensitivity(LOOKBACK_Q, h) for h in [3, 12]])
spx_full = tables['Full period']['S&P 500 B&H']
print(f"Reference, S&P 500 B&H: Sharpe {spx_full['Sharpe']:.2f}, CAGR {spx_full['CAGR']:.1%}, "
      f"MaxDD {spx_full['Max Drawdown']:.1%}")
display(sens.style.format({c: ('{:.1%}' if ('CAGR' in c or 'MaxDD' in c) else '{:.2f}')
                           for c in sens.columns if c not in ('lookback (q)', 'horizon (m)')}))

### 10b · Transaction costs and execution delay (Optimized Tiers)

In [ ]:
rows = []
for bps in [0, 5, 10, 25, 50]:
    for lag in [0, 1, 5]:
        s = perf_stats(run_backtest(STRATS['Optimized Tiers'], cost_bps=bps, exec_lag=lag)['ret'])
        rows.append({'cost (bps)': bps, 'exec lag (days)': lag, 'CAGR': s['CAGR'], 'Sharpe': s['Sharpe'],
                     'Max DD': s['Max Drawdown'], 'Calmar': s['Calmar']})
cost_tbl = pd.DataFrame(rows).pivot(index='cost (bps)', columns='exec lag (days)', values=['Sharpe', 'CAGR', 'Max DD'])
display(cost_tbl.style.format({c: '{:.2f}' if c[0] == 'Sharpe' else '{:.1%}' for c in cost_tbl.columns}))

### 10c · Crisis and rebound windows (total return)

In [ ]:
EPISODES = {'1987 crash': ('1987-08-25', '1987-12-04'), '1990 recession': ('1990-07-16', '1990-10-11'),
            'Dot-com bust': ('2000-03-24', '2002-10-09'), 'GFC': ('2007-10-09', '2009-03-09'),
            '2009 rebound (12m)': ('2009-03-10', '2010-03-09'), 'COVID crash': ('2020-02-19', '2020-03-23'),
            'COVID rebound (to YE)': ('2020-03-24', '2020-12-31'), '2022 bear': ('2022-01-03', '2022-10-12'),
            '2023-24 rally': ('2023-01-01', '2024-12-31')}
ep = pd.DataFrame({e: {n: (1 + results[n]['ret'].loc[a:b]).prod() - 1 for n in STRAT_NAMES + BENCH_NAMES}
                   for e, (a, b) in EPISODES.items() if pd.Timestamp(a) >= results['S&P 500 B&H']['ret'].index[0]
                   and pd.Timestamp(a) <= rets.index[-1]})
display(ep.style.format('{:.1%}').background_gradient(cmap='RdBu', vmin=-0.6, vmax=0.6, axis=None))

### 10d · Is the Sharpe improvement statistically meaningful?
A paired moving-block bootstrap of monthly excess returns (12-month blocks, 2 000 resamples) gives the Sharpe difference versus S&P 500 buy-and-hold. The column `P(diff ≤ 0)` is the share of resamples in which the strategy did *not* beat the benchmark's Sharpe.

In [ ]:
def boot_sharpe_diff(a, b, block=12, n=2000, seed=0):
    df = pd.concat([monthly_excess(a), monthly_excess(b)], axis=1).dropna().to_numpy()
    T = len(df)
    rng = np.random.default_rng(seed)
    starts = rng.integers(0, T - block + 1, size=(n, int(np.ceil(T / block))))
    idx = (starts[:, :, None] + np.arange(block)).reshape(n, -1)[:, :T]
    s = df[idx]                                   # (n, T, 2)
    sr = s.mean(1) / s.std(1, ddof=1) * np.sqrt(12)
    d = np.round(sr[:, 0] - sr[:, 1], 10)          # exact ties (e.g. a scaled copy) count as 'not better'
    obs = (df.mean(0) / df.std(0, ddof=1) * np.sqrt(12))
    return obs[0] - obs[1], np.percentile(d, 5), np.percentile(d, 95), (d <= 0).mean()


rows = []
for wname, (a, b) in windows.items():
    for n in STRAT_NAMES + ['S&P 10m-SMA Trend', 'Exposure-Matched Static']:
        dlt, lo5, hi95, pval = boot_sharpe_diff(results[n]['ret'].loc[a:b], results['S&P 500 B&H']['ret'].loc[a:b])
        rows.append({'window': wname, 'strategy': n, 'Sharpe diff': dlt, '5%': lo5, '95%': hi95, 'P(diff ≤ 0)': pval})
display(pd.DataFrame(rows).set_index(['window', 'strategy']).style.format('{:.2f}'))

### 10e · Look-ahead audit
Three independent checks.
1. **Truncation test.** At sample decision months the *entire* signal pipeline is rebuilt from data cut off at what was published by then: features, phases, as-of labels, the walk-forward fits and the confidence. The result must equal the full-history signal exactly. Any leak of later data shows up as a mismatch.
2. **Engine causality.** Every return after a cut-off date is replaced with garbage. Strategy returns before the cut-off must not change.
3. **Signal delay.** The same rules run on a signal delayed by 1-12 months. A genuine slow macro signal decays gradually. A result that only works with zero delay, then collapses, is a red flag for hidden timing leakage or a lucky fit.

Not testable with free data: FRED serves *revised* unemployment, not the first print. Revisions to the headline rate are small (seasonal factors, usually ≤ 0.1 pp), and fed funds is never revised. ALFRED vintages would close this gap.

In [ ]:
audit_rows = []
for d in list(conf_step.index[::36]) + [conf_step.index[-1]]:
    trunc = confidence_pipeline(macro_raw.loc[:d - PUB_LAG_M])          # only prints published by month d
    audit_rows.append({'decision month': str(d), 'full history': conf.loc[d], 'truncated data': trunc['conf'].loc[d],
                       'phase (full)': phase.loc[d], 'phase (truncated)': trunc['phase'].loc[d]})
audit = pd.DataFrame(audit_rows).set_index('decision month')
audit['abs diff'] = (audit['full history'] - audit['truncated data']).abs()
lookahead_ok = bool((audit['abs diff'] < 1e-9).all() and (audit['phase (full)'] == audit['phase (truncated)']).all())
display(audit.style.format({'full history': '{:.4f}', 'truncated data': '{:.4f}', 'abs diff': '{:.1e}'}))
print('Truncation test:', 'PASS (no future data reaches the signal)' if lookahead_ok else 'FAIL: look-ahead detected')

cut = rets.index[len(rets) // 2]
garbage = rets.copy()
garbage.loc[garbage.index > cut, ['SPX', 'NDX']] = np.random.default_rng(0).normal(0, 0.05, (int((garbage.index > cut).sum()), 2))
causal_ok = all(np.allclose(run_backtest(W_)['ret'].loc[:cut], run_backtest(W_, R=garbage)['ret'].loc[:cut])
                for W_ in STRATS.values())
print(f'Engine causality ({cut.date()} cut-off):', 'PASS' if causal_ok else 'FAIL')

delay_rows = []
common_start = pd.Period(BACKTEST_START, 'M') + 12
for k in [0, 1, 2, 3, 6, 12]:
    cs = conf_step.shift(k).loc[common_start:].dropna()
    row = {'signal delay (months)': k}
    for name, rule in [('Tiered 70/40 (user)', USER_TIERS), ('Optimized Tiers', BEST)]:
        r_ = run_backtest(alloc_tiered(cs, **rule))['ret']
        row[f'{name} Sharpe'], row[f'{name} MaxDD'] = sharpe(r_), perf_stats(r_)['Max Drawdown']
    delay_rows.append(row)
delay_tbl = pd.DataFrame(delay_rows).set_index('signal delay (months)')
display(delay_tbl.style.format({c: '{:.2f}' if 'Sharpe' in c else '{:.1%}' for c in delay_tbl.columns}))

### 10f · Overfitting checks
1. **Placebo timing test.** The confidence series is circularly shifted by every offset of at least 2 years. Each placebo has exactly the same distribution and persistence of confidence, so the same average exposure and turnover, but its *timing* is unrelated to the macro cycle. If the real signal's Sharpe does not stand out from the placebos, the result comes from the exposure profile, not from reading the cycle.
2. **Probability of Backtest Overfitting** (Bailey, Borwein, López de Prado & Zhu, 2017), via combinatorially symmetric cross-validation.
   * The 1986-2004 design window is cut into 10 blocks, and every 5-block half (252 splits) serves once as "in-sample".
   * In each split, the best of the 540 tier configurations is chosen in-sample, and its out-of-sample rank among all 540 is recorded.
   * PBO is the share of splits where the in-sample winner lands in the *bottom half* out-of-sample. Near 0 the selection is robust; near 50 % picking "the best" is no better than picking at random.
   * The right-hand scatter slopes down partly by construction: each split's two halves are complements, so a lucky in-sample half leaves a weaker out-of-sample half. Read PBO and *P(out-of-sample Sharpe < 0)* rather than the slope.

In [ ]:
def placebo_test(rule_fn, cs, min_gap=24, stride=2):
    actual = sharpe(run_backtest(rule_fn(cs))['ret'])
    v = cs.to_numpy()
    placebo = [sharpe(run_backtest(rule_fn(pd.Series(np.roll(v, k), index=cs.index)))['ret'])
               for k in range(min_gap, len(v) - min_gap, stride)]
    return actual, np.array(placebo)


PLACEBO_RULES = {'Proportional 50/50': alloc_proportional,
                 'Tiered 70/40 (user)': lambda c: alloc_tiered(c, **USER_TIERS),
                 'Optimized Tiers': lambda c: alloc_tiered(c, **BEST)}
placebo = {n: placebo_test(fn, dec) for n, fn in PLACEBO_RULES.items()}
fig, axes = plt.subplots(1, 3, figsize=(13, 3.4), sharey=True)
for ax, (n, (act, pl)) in zip(axes, placebo.items()):
    ax.hist(pl, bins=25, color=C['light'], edgecolor='white')
    ax.axvline(act, color=C['red'], lw=2)
    ax.set_title(f'{n}\nreal {act:.2f} vs placebo median {np.median(pl):.2f}  |  p = {(pl >= act).mean():.2f}',
                 fontsize=9.5)
    ax.set_xlabel('Sharpe ratio')
axes[0].set_ylabel('placebo count')
fig.suptitle('Placebo timing test: red = real signal, grey = same signal shifted in time', x=0.01, ha='left',
             fontsize=10, fontweight='bold')
fig.tight_layout()
plt.show()


def pbo_cscv(M, S=10):
    """Probability of Backtest Overfitting via CSCV. M: (months x configurations) excess returns."""
    M = M[len(M) % S:]
    blocks = np.array_split(np.arange(len(M)), S)
    finite = lambda x: np.where(np.isfinite(x), x, -1e9)
    logit, sr_is, sr_oos = [], [], []
    for combo in itertools.combinations(range(S), S // 2):
        tr = np.concatenate([blocks[i] for i in combo])
        te = np.concatenate([blocks[i] for i in range(S) if i not in combo])
        a = finite(M[tr].mean(0) / M[tr].std(0))
        b = finite(M[te].mean(0) / M[te].std(0))
        best_i = int(np.argmax(a))
        w = stats.rankdata(b)[best_i] / (M.shape[1] + 1)
        logit.append(np.log(w / (1 - w)))
        sr_is.append(a[best_i] * np.sqrt(12))
        sr_oos.append(b[best_i] * np.sqrt(12))
    logit = np.array(logit)
    return dict(pbo=float((logit <= 0).mean()), logit=logit, sr_is=np.array(sr_is), sr_oos=np.array(sr_oos))


pbo = pbo_cscv(np.column_stack(grid_m))
fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
axes[0].hist(pbo['logit'], bins=30, color=C['blue'], edgecolor='white')
axes[0].axvline(0, color=C['ink2'], lw=1)
axes[0].set_title(f"Out-of-sample rank logit of the in-sample winner (PBO = {pbo['pbo']:.0%})")
axes[0].set_xlabel('< 0: winner fell into the bottom half out-of-sample')
axes[1].scatter(pbo['sr_is'], pbo['sr_oos'], s=10, color=C['blue'], alpha=0.6)
slope = np.polyfit(pbo['sr_is'], pbo['sr_oos'], 1)[0]
axes[1].axhline(0, color=C['ink2'], lw=0.7)
axes[1].set_xlabel('in-sample Sharpe of the selected configuration')
axes[1].set_ylabel('its out-of-sample Sharpe')
axes[1].set_title(f'Performance degradation (slope {slope:+.2f}); '
                  f'P(out-of-sample Sharpe < 0) = {(pbo["sr_oos"] < 0).mean():.0%}')
fig.tight_layout()
plt.show()
print(f"PBO {pbo['pbo']:.0%} | median in-sample Sharpe of the winner {np.median(pbo['sr_is']):.2f} -> "
      f"median out-of-sample {np.median(pbo['sr_oos']):.2f} | Deflated Sharpe (selected rule) {dsr:.0%}")

## 11 · Research views
### 11a · Is training on old data a handicap? Expanding vs rolling vs time-decay training
The concern is that 1960s-80s markets were different, so old data might mislead today's model. Three facts frame the test.
* **Contractions are rare.** There are 10 since 1957 but only 4 since 1990. A model trained on the last 20 years learns from 2 or 3 episodes, so it adapts quickly but is mostly noise.
* **The features are already regime-relative.** The model never sees "fed funds = 5 %", only "fed funds at the top of its own 10-quarter range" or "unemployment up 0.4 pp in 6 months". Level shifts such as 1970s inflation, the zero lower bound or a drifting natural rate of unemployment are normalized away.
* **Extrapolating a trend in the coefficients is not recommended.** Estimates from a handful of episodes drift mostly through noise, and projecting that noise forward amplifies it. The principled way to "adapt" is to **down-weight old data** with a half-life, or to **drop it** with a rolling window. Both are tested below on the same walk-forward basis.

**Protocol.** The scheme is *chosen* on the 1986-2004 design window only, by lowest Brier score (forecast accuracy), the same hold-out rule the tier optimizer follows. The 2005-today columns then show whether that choice helped out-of-sample. The 2005+ window holds only two contractions (2008-09 and 2020), so treat small differences as ties.

In [ ]:
TRAIN_VARIANTS = {'Expanding (all history)': {}, 'Rolling 30y': dict(window_m=360), 'Rolling 20y': dict(window_m=240),
                  'Decay, half-life 20y': dict(half_life_m=240), 'Decay, half-life 10y': dict(half_life_m=120)}
TRAIN_SETTING = {'Expanding (all history)': 'TRAIN_WINDOW_M = None; TRAIN_HALF_LIFE_M = None',
                 'Rolling 30y': 'TRAIN_WINDOW_M = 360', 'Rolling 20y': 'TRAIN_WINDOW_M = 240',
                 'Decay, half-life 20y': 'TRAIN_HALF_LIFE_M = 240', 'Decay, half-life 10y': 'TRAIN_HALF_LIFE_M = 120'}
adapt_rows, adapt_coefs = [], {}
for name, kw in TRAIN_VARIANTS.items():
    kw = {'window_m': None, 'half_life_m': None, **kw}
    p_, cf_ = walk_forward(D, pipe['label_fn'], **kw)
    adapt_coefs[name] = cf_
    c_ = step(1 - p_).dropna()
    ev_ = pd.DataFrame({'p': p_, 'y': y}).dropna()
    evd, ev5 = ev_.loc[:DESIGN_END], ev_.loc[test_start:]
    auc = lambda e: roc_auc_score(e.y, e.p) if e.y.nunique() == 2 else np.nan
    row = {'training': name, 'Brier 1986-2004': brier_score_loss(evd.y, evd.p), 'AUC 1986-2004': auc(evd),
           'Brier 2005+': brier_score_loss(ev5.y, ev5.p), 'AUC 2005+': auc(ev5)}
    for lab, rule in [('User tiers', USER_TIERS), ('Optimized tiers', BEST)]:
        r_ = run_backtest(alloc_tiered(c_, **rule))['ret']
        row[f'{lab} Sharpe'], row[f'{lab} Sharpe 2005+'] = sharpe(r_), sharpe(r_.loc[test_start:])
        row[f'{lab} MaxDD'] = perf_stats(r_)['Max Drawdown']
    adapt_rows.append(row)
adapt = pd.DataFrame(adapt_rows).set_index('training')
display(adapt.style.format({c: '{:.1%}' if 'MaxDD' in c else ('{:.4f}' if 'Brier' in c else '{:.2f}')
                            for c in adapt.columns}))
train_pick = adapt['Brier 1986-2004'].idxmin()
base_ = adapt.loc['Expanding (all history)']
print(f"Picked on 1986-2004 only (lowest Brier): {train_pick}.  Out-of-sample 2005+: "
      f"AUC {adapt.loc[train_pick, 'AUC 2005+']:.2f} vs {base_['AUC 2005+']:.2f} expanding, "
      f"Optimized-tier Sharpe {adapt.loc[train_pick, 'Optimized tiers Sharpe 2005+']:.2f} vs "
      f"{base_['Optimized tiers Sharpe 2005+']:.2f}.")
if train_pick != 'Expanding (all history)':
    print(f"To adopt it for the whole notebook and the live signal, set `{TRAIN_SETTING[train_pick]}` in §1 and "
          "re-run all cells (do this only if the 2005+ columns confirm the improvement).")

fig, axes = plt.subplots(2, 1, figsize=(12, 6.4), sharex=True)
lim = np.abs(pd.concat(adapt_coefs.values())).max().max()
for ax, name in zip(axes, ['Expanding (all history)', 'Decay, half-life 10y']):
    cf_ = adapt_coefs[name].reindex(adapt_coefs['Expanding (all history)'].index)
    im = ax.imshow(cf_.T.to_numpy(), aspect='auto', cmap='RdBu_r', vmin=-lim, vmax=lim)
    ax.set_yticks(range(cf_.shape[1]), [FEATURE_LABELS[c] for c in cf_.columns], fontsize=8)
    ax.set_xticks(range(0, len(cf_), 4), [str(p.year) for p in cf_.index[::4]])
    ax.grid(False)
    ax.set_title(f'Walk-forward coefficients by refit year: {name}')
fig.colorbar(im, ax=axes, shrink=0.7, label='standardized coefficient (red raises contraction odds)')
plt.show()

# Structural-break check: same model fitted before and after 1990, scaled identically so coefficients compare.
Xa, ya, _ = trainable(D, pipe['label_fn'], D.index[-1])
split = pd.Period('1990-01', 'M')
parts = {'before 1990': (Xa.loc[:split - 1], ya.loc[:split - 1]), '1990 onward': (Xa.loc[split:], ya.loc[split:]),
         'pooled': (Xa, ya)}
brk = {k: SignedLogit(C=1e3).fit(Xk, yk, scale_ref=Xa) for k, (Xk, yk) in parts.items()}


def loglik(m, X_, y_):
    q = m.predict_proba(X_)[:, 1].clip(1e-9, 1 - 1e-9)
    return float((y_ * np.log(q) + (1 - y_) * np.log(1 - q)).sum())


lr_stat = 2 * (sum(loglik(brk[k], *parts[k]) for k in ['before 1990', '1990 onward']) - loglik(brk['pooled'], Xa, ya))
dof = Xa.shape[1] + 1
print(f'Structural break at 1990: LR = {lr_stat:.1f} on {dof} df, p = {stats.chi2.sf(lr_stat, dof):.3f}; '
      f'overlap-adjusted (labels overlap ~{HORIZON_M} months) LR = {lr_stat / HORIZON_M:.1f}, '
      f'p = {stats.chi2.sf(lr_stat / HORIZON_M, dof):.3f}')
display(pd.DataFrame({k: m.coef_ for k, m in brk.items()}).rename(index=FEATURE_LABELS).style.format('{:+.2f}'))

### 11b · Tails, kurtosis and drawdowns
Daily equity returns are fat-tailed: kurtosis far above the normal distribution's 3. Drawdowns are built from those tail days. This view asks four questions:
1. How much of each strategy's worst drawdown came from tail days (moves beyond 2.5 standard deviations)?
2. Does the regime filter cut the tails or just the average?
3. Does **rolling kurtosis** *lead* drawdowns, which would make it a usable warning signal, or does it only spike *after* the crash days have entered the window?
4. As a risk signal, does trailing kurtosis tell you more than plain trailing volatility?

This is research only; nothing here feeds the strategy.

In [ ]:
TAIL_NAMES = ['S&P 500 B&H', 'Nasdaq-100 B&H', 'Tiered 70/40 (user)', 'Optimized Tiers', 'Optimized + Trend']


def tail_profile(r):
    r = r.dropna()
    z = (r - r.mean()) / r.std()
    lr = np.log1p(r)
    nav = np.exp(lr.cumsum())
    dd = nav / nav.cummax() - 1
    trough = dd.idxmin()
    win = lr.loc[nav.loc[:trough].idxmax():trough].iloc[1:]
    worst = lr.nsmallest(max(1, len(lr) // 100)).index
    yrs = len(r) / 252
    return pd.Series({
        'Skew (daily)': stats.skew(r), 'Excess kurtosis (daily)': stats.kurtosis(r),
        'Days beyond ±3σ': (z.abs() > 3).mean(), 'Normal distribution': 2 * stats.norm.sf(3),
        'Worst day': r.min(), 'CAGR': np.exp(lr.sum() / yrs) - 1,
        'CAGR without worst 1% of days': np.exp(lr.drop(worst).sum() / yrs) - 1,
        'Max drawdown': dd.min(),
        'Max DD: share from < -2.5σ days': win[z.loc[win.index] < -2.5].sum() / win.sum(),
        'Max DD: share of its days that were < -2.5σ': (z.loc[win.index] < -2.5).mean()})


tails = pd.DataFrame({n: tail_profile(results[n]['ret']) for n in TAIL_NAMES})
ratio_rows = ['Skew (daily)', 'Excess kurtosis (daily)']
display(tails.style.format('{:.2f}', subset=pd.IndexSlice[ratio_rows, :])
        .format('{:.2%}', subset=pd.IndexSlice[[i for i in tails.index if i not in ratio_rows], :]))

# rolling kurtosis vs drawdowns on the long S&P 500 history
spx_long = rets['SPX'].loc['1950':]
roll_kurt = spx_long.rolling(63).kurt()
roll_vol = spx_long.rolling(63).std() * np.sqrt(252)
nav_l = (1 + spx_long).cumprod()
dd_l = nav_l / nav_l.cummax() - 1
episode = (dd_l == 0).cumsum()
eps = pd.DataFrame({'depth': dd_l.groupby(episode).min(), 'trough': dd_l.groupby(episode).idxmin(),
                    'peak': nav_l.groupby(episode).idxmax()})
eps = eps[eps.depth <= -0.15].nsmallest(10, 'depth')

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
axes[0].plot(roll_kurt.index, roll_kurt, color=C['violet'], lw=0.8)
axes[0].axhline(0, color=C['ink2'], lw=0.6)
axes[0].set_title('S&P 500: rolling 63-day excess kurtosis (0 = normal distribution)')
axes[1].fill_between(dd_l.index, dd_l, 0, color=C['ink2'], alpha=0.35, lw=0)
for t in eps.trough:
    for ax in axes:
        ax.axvline(t, color=C['red'], lw=0.7, alpha=0.8)
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.0%}'))
axes[1].set_title('S&P 500 drawdown; red lines = troughs of the 10 deepest drawdowns')
fig.tight_layout()
plt.show()

# event study: rolling kurtosis around drawdown peaks (start of the decline) and troughs (bottom)
K_D = np.arange(-126, 127)
pos = pd.Series(np.arange(len(roll_kurt)), index=roll_kurt.index)
fig, axes = plt.subplots(1, 2, figsize=(12, 3.6), sharey=True)
lead_lag = {}
for ax, col, title in zip(axes, ['peak', 'trough'], ['around the market PEAK (start of drawdown)',
                                                       'around the market TROUGH (bottom)']):
    paths = np.array([[roll_kurt.iloc[pos[t] + k] if 0 <= pos[t] + k < len(roll_kurt) else np.nan for k in K_D]
                      for t in eps[col]])
    for row_ in paths:
        ax.plot(K_D, row_, color=C['light'], lw=0.8)
    med = np.nanmedian(paths, axis=0)
    ax.plot(K_D, med, color=C['violet'], lw=2.2, label='median')
    ax.axvline(0, color=C['ink2'], lw=0.8)
    ax.set_title(f'63-day kurtosis {title}', fontsize=10)
    ax.set_xlabel('trading days from event')
    lead_lag[col] = int(K_D[np.nanargmax(med)])
axes[0].legend(loc='upper left')
fig.tight_layout()
plt.show()

# does trailing kurtosis predict risk better than trailing volatility? (month-end readings -> next month)
mk = pd.DataFrame({'kurt': roll_kurt, 'vol': roll_vol}).groupby(spx_long.index.to_period('M')).last()
last_m = spx_long.index[-1].to_period('M')


def max_dd_ahead(m, months=3):
    """Worst drawdown from the month-end of `m` over the following `months` months (NaN if incomplete)."""
    if m + months > last_m:
        return np.nan
    nav_ = (1 + spx_long.loc[(m + 1).start_time:(m + months).end_time]).cumprod()
    return (nav_ / nav_.cummax().clip(lower=1.0) - 1).min()


dd_ahead = pd.Series({m: max_dd_ahead(m) for m in mk.index})
nxt = pd.DataFrame({'next-month return (ann.)': mret['SPX'].shift(-1) * 12,
                    'next-month vol (ann.)': (spx_long.groupby(spx_long.index.to_period('M')).std()
                                              * np.sqrt(252)).shift(-1),
                    'P(>10% drawdown in next 3m)': (dd_ahead < -0.10).astype(float).where(dd_ahead.notna())})
q_tbl = {}
for sig in ['kurt', 'vol']:
    qq = pd.qcut(mk[sig], 5, labels=['Q1 low', 'Q2', 'Q3', 'Q4', 'Q5 high'])
    q_tbl[sig] = nxt.join(qq.rename('q')).dropna().groupby('q', observed=True).mean()
print('Next-month outcomes by quintile of TRAILING 63-day kurtosis (left) vs trailing volatility (right)')
display(pd.concat({'by kurtosis': q_tbl['kurt'], 'by volatility': q_tbl['vol']}, axis=1).style.format('{:.1%}'))

ph_kurt = pd.DataFrame({'phase': phase, 'r': mret['SPX'].shift(-1)}).loc[:DESIGN_END].dropna()
ph_kurt = ph_kurt.groupby('phase')['r'].agg(months='size', skew=lambda s: stats.skew(s),
                                            excess_kurtosis=lambda s: stats.kurtosis(s),
                                            worst=lambda s: s.min()).reindex(PHASES)
print('Shape of next-month S&P 500 returns by phase (in-sample, to 2004)')
display(ph_kurt.style.format({'months': '{:.0f}', 'skew': '{:.2f}', 'excess_kurtosis': '{:.2f}', 'worst': '{:.1%}'}))
print(f"Median 63-day kurtosis peaks {lead_lag['peak']:+d} trading days from drawdown PEAKS and "
      f"{lead_lag['trough']:+d} days from TROUGHS (negative = before the event).")

### Key findings (generated from the numbers above)

In [ ]:
def summary_line(name, t):
    s, b = t[name], t['S&P 500 B&H']
    return (f"  {name:<24} CAGR {s['CAGR']:6.1%} (S&P {b['CAGR']:.1%}) | Sharpe {s['Sharpe']:.2f} ({b['Sharpe']:.2f}) | "
            f"MaxDD {s['Max Drawdown']:6.1%} ({b['Max Drawdown']:.1%}) | Calmar {s['Calmar']:.2f} ({b['Calmar']:.2f}) | "
            f"avg exposure {s['Avg Equity Exposure']:.0%}")


for wname, t in tables.items():
    print(wname)
    for n in STRAT_NAMES:
        print(summary_line(n, t))
best_test = tables[list(windows)[2]].loc['Sharpe', STRAT_NAMES].idxmax()
print(f"\nHighest test-window Sharpe: {best_test}.")
print(f"Optimized rule: {band_text(BEST)}  (Deflated Sharpe {dsr:.0%})")
print(f"\nLook-ahead audit: truncation {'PASS' if lookahead_ok else 'FAIL'}, engine causality "
      f"{'PASS' if causal_ok else 'FAIL'}")
print(f"Overfitting: PBO {pbo['pbo']:.0%}; placebo timing p-values "
      + ', '.join(f"{n} {(pl >= act).mean():.2f}" for n, (act, pl) in placebo.items()))
print(f"Training window: design-window pick = {train_pick}; 2005+ AUC {adapt.loc[train_pick, 'AUC 2005+']:.2f} "
      f"vs expanding {adapt.loc['Expanding (all history)', 'AUC 2005+']:.2f}")
print(f"Kurtosis: the median 63-day kurtosis peak sits {lead_lag['peak']:+d} days from drawdown peaks and "
      f"{lead_lag['trough']:+d} days from troughs (negative = it leads)")

## 12 · LIVE confidence
Re-downloads the latest FRED and Yahoo data and re-fits the model on every label knowable today. It then reports:
* the current phase
* the expansion confidence, with a **90 % moving-block-bootstrap interval** that captures model-estimation uncertainty
* the 10 % step
* the allocation under each rule

The chart shows the confidence path over the last **10 quarters**.

Run the notebook top to bottom first (*Restart & Run All*), because this cell reuses the functions and the optimized tiers from above.

In [ ]:
LIVE_BOOT = 300
live_macro = load_macro(refresh=True)
live_px = fetch_yf('^GSPC', refresh=True)
live_px = live_px if live_px is not None else spx_px

Xl = to_decision_time(build_features(live_macro))
ph_l = classify_phase(Xl)
Dl = design(Xl, ph_l)
now = Dl.index[-1]
Xt, yt, wt = trainable(Dl, make_label_fn(live_macro), now, TRAIN_WINDOW_M, TRAIN_HALF_LIFE_M)
model_l = fit_model(Xt, yt, wt)
path_idx = Dl.index[-LOOKBACK_M:]
point = pd.Series(1 - model_l.predict_proba(Dl.loc[path_idx])[:, 1], index=path_idx)

rng = np.random.default_rng(42)
T, blk = len(yt), 24
boots = []
while len(boots) < LIVE_BOOT:
    ix = (rng.integers(0, T - blk + 1, int(np.ceil(T / blk)))[:, None] + np.arange(blk)).ravel()[:T]
    if yt.iloc[ix].nunique() < 2:
        continue
    boots.append(1 - fit_model(Xt.iloc[ix], yt.iloc[ix], wt[ix]).predict_proba(Dl.loc[path_idx])[:, 1])
boots = np.array(boots)
band_lo, band_hi = np.percentile(boots, 5, axis=0), np.percentile(boots, 95, axis=0)

c_now, c_step = point.iloc[-1], float(step(pd.Series([point.iloc[-1]])).iloc[0])
px_m = live_px.groupby(live_px.index.to_period('M')).last()
sma = px_m.iloc[-TREND_SMA_M:].mean()
trend_now = bool(px_m.iloc[-1] > sma)
c_trend = max(c_step, BEST['lo']) if trend_now else c_step

last_obs = {k: live_macro[k].last_valid_index() for k in ['UNRATE', 'FEDFUNDS']}
row = Xl.iloc[-1]
print('=' * 78)
print(f"  LIVE MACRO REGIME FILTER   run {pd.Timestamp.today():%Y-%m-%d}   decision month {now}")
print(f"  data: UNRATE through {last_obs['UNRATE']}, FEDFUNDS through {last_obs['FEDFUNDS']}, "
      f"S&P through {live_px.index[-1]:%Y-%m-%d}")
print('=' * 78)
print(f"  Phase                 : {ph_l.iloc[-1]}")
print(f"  Expansion confidence  : {c_now:.1%}   (90% CI {band_lo[-1]:.1%} to {band_hi[-1]:.1%})")
print(f"  Stepped confidence    : {c_step:.0%}")
print(f"  P(contraction, next {HORIZON_M}m): {1 - c_now:.1%}")
print(f"  Unemployment {row['ur']:.1f}%  (6m chg {row['ur_trend']:+.2f}pp, Sahm gap {row['ur_gap']:.2f}pp, "
      f"{row['ur_pos']:.0%} of {LOOKBACK_Q}q range)")
print(f"  Fed funds    {row['ff']:.2f}% (6m chg {row['ff_trend']:+.2f}pp, {row['ff_off_peak']:+.2f}pp vs {LOOKBACK_Q}q high, "
      f"{row['ff_pos']:.0%} of range)")
print(f"  S&P 500 vs {TREND_SMA_M}m SMA   : {'ABOVE' if trend_now else 'BELOW'} ({px_m.iloc[-1] / sma - 1:+.1%})")
print('-' * 78)

live_alloc = {
    'Proportional 50/50': alloc_proportional(pd.Series([c_step])).iloc[0],
    'Tiered 70/40 (user)': alloc_tiered(pd.Series([c_step]), **USER_TIERS).iloc[0],
    'Optimized Tiers': alloc_tiered(pd.Series([c_step]), **BEST).iloc[0],
    'Optimized + Trend': alloc_tiered(pd.Series([c_trend]), **BEST).iloc[0],
}
live_tbl = pd.DataFrame(live_alloc).T
live_tbl['allocation'] = [describe_alloc(r_.SPX, r_.NDX) for r_ in live_tbl.itertuples()]
display(live_tbl.style.format({'SPX': '{:.0%}', 'NDX': '{:.0%}', 'CASH': '{:.0%}'}))

fig, axes = plt.subplots(2, 1, figsize=(12, 6.4), gridspec_kw=dict(height_ratios=[0.9, 2.2]))
ax = axes[0]
edges = [0, BEST['lo'], BEST['hi'], 1.0]
labels = [describe_alloc(*BEST['low']), describe_alloc(*BEST['mid']), describe_alloc(*BEST['top'])]
for (a, b), col, lab in zip(zip(edges[:-1], edges[1:]), [C['red'], C['yellow'], C['aqua']], labels):
    ax.axvspan(a, b, color=col, alpha=0.18, lw=0)
    ax.text((a + b) / 2, 0.82, lab, ha='center', va='center', fontsize=8, color=C['ink2'])
ax.errorbar([c_now], [0.4], xerr=[[c_now - band_lo[-1]], [band_hi[-1] - c_now]], fmt='o', color=C['ink'],
            ms=9, capsize=5, lw=1.6)
ax.annotate(f'{c_now:.0%}  (step {c_step:.0%})', (c_now, 0.4), xytext=(0, -16), textcoords='offset points',
            ha='center', fontsize=9, fontweight='bold')
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_yticks([])
ax.grid(False)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.0%}'))
ax.set_title(f'Live expansion confidence with 90 % bootstrap interval, phase: {ph_l.iloc[-1]} (Optimized tiers shown)')
ax = axes[1]
t_ = path_idx.to_timestamp()
ax.fill_between(t_, band_lo, band_hi, color=C['blue'], alpha=0.15, lw=0, label='90 % bootstrap band')
ax.plot(t_, point, color=C['blue'], marker='o', ms=4, label='confidence')
ax.step(t_, step(point), where='post', color=C['ink2'], lw=1.0, label='10 % step')
for v in [BEST['lo'], BEST['hi']]:
    ax.axhline(v, color=C['ink2'], lw=0.7)
ax.set_ylim(0, 1.02)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.0%}'))
ax.set_title(f'Confidence path, last {LOOKBACK_Q} quarters')
ax.legend(loc='lower left', ncol=3)
fig.tight_layout()
plt.show()
print('Research tool, not investment advice. The live reading uses the latest prints, which may be revised.')